# Cetacean Strandings from Space: Comparing counts

This workbook is designed to compare the analysis of multiple observers counts in satellite and aerial imagery.

Specifically, the workbook is built to compare satellite counts annotated using the Cetacean Strandings from Space standardised workflow(s) and attributes, available at: https://doi.org/10.1016/j.mex.2026.103949

The code expects files to be named using the following standardised naming convention: 'location_satellite-sensor_image-id_image-date_image-time_resolution_observer-id.csv (_e.g.,_ chatham-island_geoeye1_105001002F60AA00_20221014_213434_50_1.csv).

The aerial imagery count comparison is designed to recieve annotations made using VGG. Annotation guidance available in Cubaynes et al., (2024)
(https://zslpublications.onlinelibrary.wiley.com/doi/10.1002/rse2.391, Supporting Material Data S1). Aerial files should be named location_aerial_observer-id.csv _e.g.,_ chatham_island_aerial_1.csv

The code enables plotting of total counts and counts filtered by certainty of a detection. The code further enables comparison between multi-observers annotations through hierarchical clustering, using an adapted version of code from Attard, M., (2025) (https://doi.org/10.3354/esr01396).

The semi-automated approach to clustering aims to increase the efficiency of evaluating count congruence. The clustering algorithm applies Wards method to amass points within a user-defined distance threshold, by calculating pairwise distances between individual points to form each cluster, and iteratively merging pairs of clusters to minimise within cluster variance. To improve clustering accuracy, created clusters are further refined using a series of constraints, whereby clusters must not exceed the maximum number of observers and each unique observer must only occur once. The centroid of each cluster is computed and the furthest duplicate of an observer is removed and re-clustered within a distance threshold. Re-clustering ensures to exclude the current cluster and any existing clusters that exceed the maximum number of observers, and those already containing the unique observer, else a new cluster is created. The median of all points within a cluster is then calculated and plotted to represent the cluster location. 

Clusters must then be reviewed manually to verify accuracy, and corrected if needed. To assess the accuracy of automated clustering, the code enables calculation of a Rand index (RI) for each image, which quantifies how well the automated clustering performs in comparison to the corrected clustering (1 = near perfect agreement, 0 = no agreement). The code also calculates an adjusted Rand index (ARI), a more robust measure of cluster performance, which eliminates clusters achieved by random guessing (1 = perfect match, 0 = no better than random clustering, -1 = worse than random clustering). 

**Python tips**

For anyone new to Python/coding, here are some useful tips to navigate Jupyter Notebooks:
- when a cell is blue, it is command mode (binding the keys to notebook level commands, when in command mode press h to view all the notebook level commands), when a cell is green, it is in edit mode (allowing you to type code and text in a cell)
- to run a code cell, double click a cell (or when the cell is blue press the enter key) to enter edit mode, press the shift + enter keys
- to create a new cell, (in command mode) pressing 'a' inserts a new cell above and 'b' inserts a new cell below
- to save the notebook (if you are in edit mode, press the 'esc' key to exit, the cell will turn blue) and press the 's' key
- jupyter where possible will auto-complete your code, begin typing and press tab
- hash # indicates comments to help you with code

## Importing python packages

In [ ]:
# import Python libraries required throughout the workbook
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import os
import glob
from collections import defaultdict
import json
import re
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D  # import Line2D to create custom legend handles
import seaborn as sns
import numpy as np
from PIL import Image, ImageDraw
from sklearn.metrics import adjusted_rand_score
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.special import comb
import rasterio
from rasterio.warp import transform
from rasterio.crs import CRS
from pyproj import CRS, Transformer
from skimage import exposure

# Import tested helper functions from the local strandings_from_space package.
from strandings_from_space.filenames import (
    get_observer,
    get_resolution,
    group_by_indices,
    group_by_indices_excluding_observer as group_by_indices_exc_observer,
)


## Set the working directories

In [ ]:
# set working directory to your desktop
# the code automatically selects your desktop
# for another location please amend the file path for main_path below
# get user's home directory
home_directory = os.path.expanduser('~')

# append the desktop folder to the home directory
desktop_directory = os.path.join(home_directory, 'Desktop')

# set and store the desktop as the working directory
os.chdir(desktop_directory)
print(f"The current working directory is set to: {os.getcwd()}")
main_path = os.getcwd()

# confirm whether the main_path exists
if os.path.exists(main_path):
    print(main_path, 'is a correct path'),
else:
    print(main_path, 'is not a correct path')

In [ ]:
# create a number of folders to host the inputs and outputs of the code run through this workbook
folders = [
    'strandings_from_space/compare_counts/inputs',
    'strandings_from_space/compare_counts/inputs/satellite/counts',
    'strandings_from_space/compare_counts/inputs/satellite/images',
    'strandings_from_space/compare_counts/inputs/aerial_ref/counts',
    'strandings_from_space/compare_counts/inputs/aerial_ref/images',
    'strandings_from_space/compare_counts/inputs/ground_ref',
    'strandings_from_space/compare_counts/outputs',
    'strandings_from_space/compare_counts/outputs/clusters',
    'strandings_from_space/compare_counts/temp_outputs'
]

# check if the directory folder exists or not
# if the directory is not present, then create it
for folder in folders:
    os.makedirs(os.path.join(main_path, folder), exist_ok=True)

In [ ]:
# set the working path directories required throughout
input_path = os.path.join(main_path,'strandings_from_space\\compare_counts\\inputs')
input_satellite_counts = os.path.join(main_path,'strandings_from_space\\compare_counts\\inputs\\satellite\\counts')
input_satellite_images = os.path.join(main_path,'strandings_from_space\\compare_counts\\inputs\\satellite\\images')
input_aerial_counts = os.path.join(main_path,'strandings_from_space\\compare_counts\\inputs\\aerial_ref\\counts')
input_aerial_images = os.path.join(main_path,'strandings_from_space\\compare_counts\\inputs\\aerial_ref\\images')
input_ground = os.path.join(main_path,'strandings_from_space\\compare_counts\\inputs\\ground_ref')
output_path = os.path.join(main_path,'strandings_from_space\\compare_counts\\outputs')
temp_output_path = os.path.join(main_path,'strandings_from_space\\compare_counts\\temp_outputs')

Define a colour palette to be used throughout plotting:

In [ ]:
# define the colorblind-friendly palette RGB values
colorblind_palette = {
    'blue': (0.0, 0.45, 0.70),
    'orange': (0.87, 0.60, 0.0),
    'sky_blue': (0.35, 0.70, 0.90),
    'blueish_green': (0.0, 0.62, 0.45),
    'yellow': (0.95, 0.90, 0.25),
    'blueish_purple': (0.8, 0.4, 0.0),
    'vermillion': (0.8, 0.4, 0.0),
    'reddish_purple': (0.8, 0.6, 0.7)
}

## Satellite analysis

### Load satellite count .csv files

In [ ]:
# import all .csv files containing the counts for each observer in satellite imagery
# expects naming format of the annotation metadata template from the 'Cetacean Strandings from Space' workflow available here: https://doi.org/10.1016/j.mex.2026.103949
# naming convention: 'location_satellite-sensor_image-id_image-date_image-time_resolution_observer-id.csv
input_satellite_csv = glob.glob(os.path.join(input_satellite_counts, '*.csv'))
print(input_satellite_csv)

### Load satellite image .tif files

In [ ]:
# load all satellite imagery in input_satellite_images that are .tif format using glob
input_satellite_img_tif = glob.glob(os.path.join(input_satellite_images, '*.tif'))
print(input_satellite_img_tif)

### Clean and prepare .csv data

In [ ]:
# access each .csv file and store in a list
list_csv_name = []
for file in input_satellite_csv:
    dir_csv,file_csv = os.path.split(file)
    file_path = file_csv
    list_csv_name.append(file_path)
print(list_csv_name)

# define a function to group .csv filename by separator indices 5 ('gsd_m') and 2 ('img_cat_id')
# Function `group_by_indices` is imported from strandings_from_space.filenames.
# group the filenames by spatial resolution and image id
grouped_csv_name = group_by_indices(list_csv_name, separator='_', indices=[5, 2])
# grouped_csv_name = group_by_indices(list_csv_name)  # same result as above
print(grouped_csv_name)

# convert groups 'gsd_m' and 'img_cat_id' to a list of keys
group_csv_keys = list(grouped_csv_name.keys())
print(group_csv_keys)

In [ ]:
# convert grouped .csv filenames to a pandas dataframe
# first store all .csv files to open, as a list
csv_to_open = []
for key, files in grouped_csv_name.items():
    for file in files:
        csv_to_open.append({'group': key, 'file': file})

# for file in list csv_to_open, create a pandas dataframe and append to csv_df_list
csv_df_list = []
for item in csv_to_open:
    key = item['group']
    file = item['file']
    csv_df = pd.read_csv(os.path.join(input_satellite_counts, file))
    print(f"df '{file}' created")
    csv_df_list.append(csv_df)

# create a copy of the original dataframe to preserve the data
csv_df_list_copy = []
for df, item in zip(csv_df_list, csv_to_open):
    csv_df_copy = df.copy()
    print(f"csv_df_copy '{item['file']}' created")
    csv_df_list_copy.append(csv_df_copy)

# create a function
# get the value after the last underscore in the filename corresponding to the 'observer'
# Function `get_observer` is imported from strandings_from_space.filenames.
# it is important to process the data with anonymity
# ensure the filename observer-id is a number assigned to each observer
# if  names of observers are used in the original attribute table
# for each dataframe in the list of dataframes modify the column 'observer' if the condition is met
# for example, if the observer-id in the filename is 1
# amend the observer name in the column 'observer' to the value 1
# repeat with elif for the number of observers
for df, item in zip(csv_df_list_copy, csv_to_open):
    file_value = get_observer(item['file'])
    if file_value == '1':
        # change all values in 'observer' to 'new_value'
        df['observer'] = file_value
        print(f"Modified DataFrame from file '{item['file']}'")
    elif file_value == '2':
        # change all values in 'observer' to 'new_value'
        df['observer'] = file_value
        print(f"Modified DataFrame from file '{item['file']}'")
    elif file_value == '3':
        # change all values in 'observer' to 'new_value'
        df['observer'] = file_value
        print(f"Modified DataFrame from file '{item['file']}'")

    # print(df['observer'])

# change the data type of the observer column to catagorical data
# print the modified dataframes to view the observer column has amended correctly
for df in csv_df_list_copy:
    df['observer'] = pd.Categorical(df['observer'])
    print(f"data type of column observer is {df['observer'].dtype}")
    print(df)

In [ ]:
# if only the primary observer has completed all columns in the attribute table:
# it is necessary to populate certain empty columns for the remaining observers
# for example, you will need to populate the spatial resolution in column 'gsd_m'
# create a function
# get the value after the second to last underscore in the filename corresponding to 'gsd_m'
# Function `get_resolution` is imported from strandings_from_space.filenames.
# iterate over the dataframe and modify the column 'gsd_m' if the condition is met
# for example, where the filename is equal to 50 at the second to last value
# then populate the column gsd_m with 0.5
# repeat for all gsd_m as required
for df, item in zip(csv_df_list_copy, csv_to_open):
    resolution = get_resolution(item['file'])
    if resolution == '50':
        # change all values in 'gsd_m' to 'new_value'
        df['gsd_m'] = 0.5
        print(f"Modified DataFrame from file '{item['file']}'")
    elif resolution == '30':
        # change all values in 'gsd_m' to 'new_value'
        df['gsd_m'] = 0.3
        print(f"Modified DataFrame from file '{item['file']}'")
    elif resolution == '15':
        # change all values in 'gsd_m' to 'new_value'
        df['gsd_m'] = 0.15
        print(f"Modified DataFrame from file '{item['file']}'")
    elif resolution == '28':
        # change all values in 'gsd_m' to 'new_value'
        df['gsd_m'] = 0.28
        print(f"Modified DataFrame from file '{item['file']}'")

# print the modified dataframes to see the changes to column 'gsd_m' have been made correctly
for df in csv_df_list_copy:
    print(df)

In [ ]:
# for of list of dataframes (csv_df_list_copy)
# and names of files to open (csv_to_open)
# print the groups
# groups should correspond to spatial resolution/'gsd_m' and image catalogue id 'img_cat_id'
for df, item in zip(csv_df_list_copy, csv_to_open):
    print(item['group'])

In [ ]:
# concatenate the individual observers dataframes for each group

# create a dictionary to hold lists of dataframes by group
grouped_csv_dfs = {}

# for dataframe in zip(csv_df_list_copy, csv_to_open)
# append each dataframe to the corresponding group 'gsd_m' and 'img_cat_id'
for df, item in zip(csv_df_list_copy, csv_to_open):
    group = item['group']
    if group not in grouped_csv_dfs:
        grouped_csv_dfs[group] = []
    grouped_csv_dfs[group].append(df)

# use pd.concat to concatenate the dataframes for each group
# this is necessary to create a single dataframe
# containing all observers for each image catalogue id at each spatial resolution
concatenated_csv_dfs = {group: pd.concat(dfs, ignore_index = True) for group, dfs in grouped_csv_dfs.items()}

# add a new column to each concatenated dataframe and provide each each point in the dataframe a unique id number
# this is important for further in the code, to calculate the Rand Index and Adjusted Rand Index of the clustering performance
# print the concatenated dataframes to view the dataframes were concatenated correctly
# each observers dataframe should be stacked on top of each other per group
for group, df in concatenated_csv_dfs.items():
    df['unique_id'] = range(len(df))
    print(f"Concatenated DataFrame for group '{group}':")
    print(df)
    clean_group = "_".join(map(str, group))
    df.to_csv(os.path.join(temp_output_path, f'{clean_group}_concatenated_data.csv'), index=False)

In [ ]:
# create a dictionary to hold all unique values for each observer per group
unique_sat_observer_values = {}

# use set to store unique observer IDs across all groups
unique_observer_ids_set = set()

# for each dataframe of each group of concatenated_csv_dfs
# get the unique values in 'observer' column
for group, df in concatenated_csv_dfs.items():
    unique_sat_observer = df['observer'].unique()
    unique_sat_observer_values[group] = unique_sat_observer

    # add the unique observer IDs from this group to the set
    unique_observer_ids_set.update(unique_sat_observer)

# convert the set of unique observer IDs to a sorted list
# (optional: remove `sorted` if the order doesn't matter)
unique_observer_ids = sorted(unique_observer_ids_set)

# print the unique values of observer for each group 'gsd_m' and 'img_cat_id'
# for each group the unique values of observers should be equal to the total number of observers
# for example, three observers ['1' '2' '3']
for group, unique_sat_observer in unique_sat_observer_values.items():
    print(f"Unique values in 'group' for observer '{group}': {unique_sat_observer}")

# print the unique observer IDs for all groups
print("Unique observer IDs:", unique_observer_ids)

In [ ]:
# create a dictionary that groups all concatenated dataframes by the 'img_cat_id' only
# first create a dictionary to hold lists of dataframes for each second value of the group
# in this case 'img_cat_id'
grouped_by_img = {}

# loop over concatenated_csv_dfs and group by the second value 'img_cat_id'
for group, df in concatenated_csv_dfs.items():
    img_cat_id = group[1]
    if img_cat_id not in grouped_by_img:
        grouped_by_img[img_cat_id] = []
    grouped_by_img[img_cat_id].append(df)

# use pd.concat to concatenate the dataframes for each second value 'img_cat_id'
concatenated_by_img = {img_cat_id: pd.concat(dfs, ignore_index=True) for img_cat_id, dfs in grouped_by_img.items()}

# print the concatenated dataframes
for img_cat_id, df in concatenated_by_img.items():
    print(f"Concatenated DataFrame for image id '{img_cat_id}':")
    print(df)

In [ ]:
# detail the unique values of spatial resolution gsd_m' per image catalogue id 'img_cat_id'
# create a dictionary to hold unique values for each 'img_cat_id'
unique_gsd_m_values = {}

# loop over concatenated_by_img and get unique values in 'gsd_m' column
for img_cat_id, df in concatenated_by_img.items():
    unique_values = df['gsd_m'].unique()
    unique_gsd_m_values[img_cat_id] = unique_values

# print the unique values for each img_id
for img_cat_id, unique_values in unique_gsd_m_values.items():
    print(f"Unique values in 'gsd_m' for img_id '{img_cat_id}': {unique_values}")

In [ ]:
# https://datagy.io/pandas-count-unique-values-groupby/
# get unique value counts for each observer within each 'gsd_m' group
# first create a dictionary to store the resulting dataframes
total_counts_csv_dfs = {}

# loop over concatenated_by_img
# get unique value counts for each observer within each 'gsd_m' group
for img_cat_id, df in concatenated_by_img.items():
    print(f"Unique value counts for img_cat_id '{img_cat_id}':")
    total_csv_counts = df.groupby('gsd_m')['observer'].value_counts()
    total_counts_csv_df = total_csv_counts.reset_index(name='counts')
    print(total_counts_csv_df)
    # store the dataframe in the dictionary total_counts_csv_dfs
    total_counts_csv_dfs[img_cat_id] = total_counts_csv_df

# for img_cat_id, total_counts_df in total_counts_dfs.items():

### Load reference / ground count data

In [ ]:
# load in a .csv file containing the ground reference count
# for comparison with satellite or aerial data
# the column format should be:
# 'sensor' (value either 'satellite' or 'aerial')
# 'image_id' (value either the 'img_cat_id' of the satellite or matched aerial image id/name)
# 'ref_value' (integer value of the ground reference/count)
input_ground_ref_files = glob.glob(os.path.join(input_ground, '*.csv'))
print(input_ground_ref_files)

In [ ]:
# open the ground reference count .csv as a pandas dataframe and append to list ref_list_by_sensor
# create a empty list to store the opened dataframe containing the ground reference count
ref_list_by_sensor = []
# loop through any .csv files loaded from the input_ground path
# and covert to a dataframe and append to ref_list_by_sensor
for file in input_ground_ref_files:
    ref = pd.read_csv(file)
    ref_list_by_sensor.append(ref)
    print(ref_list_by_sensor)

In [ ]:
# group the dataframes in ref_list_by_sensor by ‘sensor’ column and store the grouped objects in a dictionary
# for example the file could be one document containing satellite or aerial data ground reference comparison counts
# to select only the satellite reference counts for comparing with satellite counts, we need to group by sensor
# first create an empty dictionary to store the grouped by sensor ground reference dataframes
groupby_ref = {}

# loop through the dataframe in ref_list_by_sensor, groupby 'sensor' type and add to the dictionary groupby_ref
# enumerate(ref_list_by_sensor) provides both the index (item) and the dataframe (df) from the list ref_list_by_sensor
for item, df in enumerate(ref_list_by_sensor):
    # grouby 'sensor' column
    grouped_sensor = df.groupby('sensor')
    # groupby 'sensor' object is stored in the dictionary groupby_ref with the index item as the key
    groupby_ref[item] = grouped_sensor

In [ ]:
# for each sensor type in groupby_ref loop through to select only the 'sensor' satellite to compare with satellite counts
# first create two empty dictionaries, ref_grouped_by_sensor and ref_satellite_list
ref_grouped_by_sensor = {}
ref_satellite_list = {}
# for the index and groupby 'sensor' object in groupby_ref
for key, grouped in groupby_ref.items():
    # loop through the grouby object to get the 'sensor' (name of group) and df (the dataframe for that group)
    for sensor, df in grouped:
        # convert the dataframe to a dictionary where the 'img_id' is the key and 'ref_value' is the value
        ref_grouped_dict = df.set_index('img_id')['ref_value'].to_dict()
        ref_grouped_by_sensor[(key, sensor)] = ref_grouped_dict # ref_grouped_dict is stored in ref_group_by_sensor with a tuple (key, sensor) as the key
        # print the sensor and dataframe
        print("sensor:", sensor)
        print("dataframe:", df)
        if sensor == 'satellite':
            ref_satellite_list.update(ref_grouped_dict)  # update ref_satellite_list if the sensor is ‘satellite’

# print the dictionaries to view the data has grouped correctly
print('ref_grouped_by_sensor:', ref_grouped_by_sensor) # dictionaries of reference values indexed by image ids, grouped by sensor and key
print('ref_satellite_list:', ref_satellite_list) # reference values specifically for the ‘satellite’ sensor

In [ ]:
# to simplify the above you could always define the dictionary ref_satellite_list in code rather than load a .csv
# ref_list = {
#     'DS-PHR1B-201811282257268-FR1-PX-E167S47-0909-00925': 145,
#     '1030010089B22D00': 145,
#     '105001002F0CA400': 14,
#     '105001002F60AA00': 232,
#     '10300100DC306300': 232,
#     '104001007E5E7400': 232,
#     '10300100DB012A00': 245
# }

# print(ref_list)

### Define native spatial resolution 'gsd_m'

Satellite sensors collect imagery at a certain spatial resolution. The expected and the realised spatial resolution, can differ slightly due to nadir angle, which can alter the ground sampling distance of a satellite collection. Satellite companies are now offering proprietary algorithms that can artificially enhance the spatial resolution of imagery. For example, Vantor (formerly Maxar Technologies Ltd), offer a service where satellite sensors at 0.5 m native resolution can be enhanced to 0.3 m resolution and 0.3 m native resolution can either be downsampled to 0.5 m resolution or enhanced to 0.15 m resolution. Artificially amending the resolution could create artefacts that are no representative of reality on the ground, impacting counts. Therefore it could be important to detail in the plots comparing counts whether the spatial resolution is native or artificially enhanced or down sampled, to see whether there are differences between counts, for example at native 0.3 m, compared with enhanced 0.3 m imagery.

In [ ]:
# create a .csv and store in the inputs folder containing the columns:
# img_cat_id (the image catalogue id of your satellite imagery)
# and 'native_resolution' (what the native resolution of the sensor is that captured the satellite image)
# the native resolution can be found by searching the sensor on the providers website
# load in a .csv file containing details of the native resolution of the satellite imagery
native_res_path = os.path.join(input_path, 'native_resolution.csv')
print(native_res_path)

In [ ]:
# convert the .csv to a pandas dataframe
native_res = pd.read_csv(native_res_path)
print(native_res)

### Plotting

For each image catalogue id, plot total counts per observer, at each spatial resolution:

In [ ]:
# for each img_cat_id, plot the total counts per observer at each 'gsd_m' available
# set grid style
sns.set(style='whitegrid') # style='white' for no gridlines

# loop over total_counts_csv_dfs and plot the data for each group
for (img_cat_id, total_counts_csv_df) in total_counts_csv_dfs.items():

    # initialise a img_filename variable
    img_filename = None

    # loop over grouped_csv_name to find the matching image_cat_id
    for (gsd_m, img_id), filename in grouped_csv_name.items():
        if img_id == img_cat_id:
            img_filename = filename[0][:-9]

    # calculate the y axis maximum for the current dataframe
    axis_max = total_counts_csv_df['counts'].max() + 10

    # create a figure
    plt.figure(figsize=(10, 6))

    # ensure 'gsd_m' and 'native_resolution' are of the same type
    native_res_value = native_res.loc[native_res['img_id'] == img_cat_id, 'native_resolution_m'].values
    native_res_value = native_res_value.item() # convert array to scalar

    # filter the data for native resolution
    native_res_df = total_counts_csv_df[total_counts_csv_df['gsd_m'] == native_res_value].copy() # use .copy() to avoid SettingWithCopyWarning

    # for each img_cat_id plot the total count per observer for each gsd_m with pointplot
    ax = sns.pointplot(
        x = 'gsd_m',
        y = 'counts',
        hue = 'observer', # colours based on number of observers
        markers = ['s', 'o', '^'], # define the marker style for each observer
        linestyles = ['-', '--', '-.'], # define the line plotting style for each observer
        palette = 'colorblind',
        data = total_counts_csv_df,
        order = sorted(total_counts_csv_df['gsd_m'].unique(), reverse = True) # include only present gsd_m values in descending order
    )

    # capture x-axis ticks and limits from the dataset
    x_ticks = ax.get_xticks() # get the x-ticks from the dataset
    x_limits = ax.get_xlim() # get the x-axis limits

    # set all markers to no fill (outline only)
    for line in ax.lines:
        line.set_markerfacecolor('none')  # set no fill for the marker
        line.set_markeredgecolor(line.get_color())  # set edge colour to match line colour

    # create a new dataframe for the plot to overlay total counts associated with native resolution as filled points, so they are easily identifiable
    # ensure to include all gsd_m values
    native_plotting_data = total_counts_csv_df.copy()
    native_plotting_data['counts'] = native_plotting_data['counts'].where(native_plotting_data['gsd_m'] == native_res_value, float('nan'))

    # overlay the modified data (native resolution + NaN counts for others) using pointplot with no lines
    sns.pointplot(
        x = 'gsd_m',
        y = 'counts',
        hue = 'observer', # different color based on observer
        data = native_plotting_data, # modified data with NaN counts for non-native resolutions
        markers = ['s', 'o', '^'], # assign markers for each observer
        palette = 'colorblind',
        linestyles = ['','',''], # no lines between points
        ax = ax,
        dodge = False, # disable horizontal adjustment to avoid overlap of points
        legend = True # enable the legend for scatterplot to add to legend
    )

    # set x-ticks and x-limits to ensure alignment
    ax.set_xticks(x_ticks) # reapply the x-ticks from the original pointplot
    ax.set_xlim(x_limits) # lock the x-axis limits to match the dataset

    # determine the y-axis limit by which is larger, the total count max or ground reference count
    if img_cat_id in ref_satellite_list:
        ref_value = ref_satellite_list[img_cat_id]
        y_max = max(axis_max, ref_value + 10)
        ax.axhline(ref_value, color='0.5', linestyle='--', label='Reference: {ref_value}')
        ax.axhspan(0, ref_value, color='0.7', alpha=0.3)
    else:
        y_max = axis_max

    # combine legend handles and labels manually after plotting scatterplot
    handles, labels = ax.get_legend_handles_labels()

    # print the handles to understand what is being generated is correct
    print("Handles before filtering:", handles)
    print("Labels before filtering:", labels)

    # condition to check if there is only the native resolution
    unique_gsd_m = total_counts_csv_df['gsd_m'].nunique()

    # modify handles and labels based on conditions
    if unique_gsd_m == 1:  # only the native resolution is present
        # only include the second native point plot handles, exclude the first set which is for the original line plot
        observer_count = len(total_counts_csv_df['observer'].unique())

        # assuming the first 'observer_count' handles correspond to the line plot
        # and the remaining handles correspond to second plot (native resolution)
        handles = handles[observer_count:]  # skip the line plot handles

        # define the labels for native resolution only
        labels = [
            'Observer 1: Native Resolution', 
            'Observer 2: Native Resolution', 
            'Observer 3: Native Resolution'
        ]

        # include the reference label if available
        if img_cat_id in ref_satellite_list:
            labels.append(f'Reference: {ref_value}')
    else:
        # multiple resolutions, include both line plot and native resolution point plot in the legend
        if img_id in ref_satellite_list:
            labels = [
                'Observer 1', 'Observer 2', 'Observer 3', 
                'Observer 1: Native Resolution', 'Observer 2: Native Resolution', 
                'Observer 3: Native Resolution', f'Reference: {ref_value}'
            ]
        else:
            labels = [
                'Observer 1', 'Observer 2', 'Observer 3', 
                'Observer 1: Native Resolution', 'Observer 2: Native Resolution', 
                'Observer 3: Native Resolution'
            ]

    # print the updated handles and labels to verify
    print("Handles after filtering:", handles)
    print("Labels after filtering:", labels)

    # call ax.legend() after modifying handles and labels
    # anchor the legend to a specified location
    ax.legend(handles=handles[:len(labels)], labels=labels, bbox_to_anchor=(1.05, 1), loc='upper left')

    # set the axis limit and amend the axis names
    ax.set(ylim = (0, y_max))
    plt.xlabel('Image Resolution (m)')
    plt.ylabel('Total Count (No. of Individuals)')
    plt.title(f'Total count: {img_filename}')

    # update the x-axis labels to bold and asterisk the gsd_m which is the native resolution for the particular sensor
    x_labels = ax.get_xticklabels()
    new_labels = []
    for label in x_labels:
        text = label.get_text()
        if text and float(text) == native_res_value:
            new_labels.append(r'$\mathbf{' + text + r'^*}$')
        else:
            new_labels.append(text)
    ax.set_xticklabels(new_labels)

    # save the plot
    file_path = os.path.join(output_path, f'{img_filename}_total_count.png')
    print(f"Saving plot to: {file_path}")
    plt.savefig(file_path, bbox_inches='tight')
    # plt.savefig(os.path.join(output_path, f'Plot for image:{img_filename}.png'), dpi=100)

    # show the plot
    plt.show()

    # close the figure to avoid memory issues
    plt.close()

Filtering the data by certainty to get total counts where the data is only definite or likely stranded whale:

In [2]:
# get the total counts for each observer filtered to remove counts where 'certainty' was assigned 'possible_50-70'
# https://datagy.io/pandas-count-unique-values-groupby/
# create an empty dictionary to store the resulting dataframes
filtered_sat_counts_dfs = {}

# loop over concatenated_by_img and filter each dataframe to remove 'possible_(50-769)' values from the '*certainty' column
for img_cat_id, df in concatenated_by_img.items():
    df_filtered_certainty = df[df['certainty'] != 'possible_50-69']

    # get the new total count per observer less of 'possible_50-69'
    filtered_sat_total_counts = df_filtered_certainty.groupby('gsd_m')['observer'].value_counts().reset_index(name='counts')

    # store the resulting filtered total count dataframes in the dictionary filtered_sat_counts_dfs
    filtered_sat_counts_dfs[img_cat_id] = filtered_sat_total_counts

print(filtered_sat_counts_dfs)

In [ ]:
# for each img_cat_id, plot the total counts per observer at each 'gsd_m' available
# set grid style
sns.set(style='whitegrid') # style='white' for no gridlines

# loop over filtered_sat_counts_dfs and plot the data for each group
for (img_cat_id, filtered_sat_total_counts) in filtered_sat_counts_dfs.items():

    # initialise a img_filename variable
    img_filename = None

    # loop over grouped_csv_name to find the matching image_cat_id
    for (gsd_m, img_id), filename in grouped_csv_name.items():
        if img_id == img_cat_id:
            img_filename = filename[0][:-9]

    # calculate the y axis maximum for the current dataframe
    axis_max = filtered_sat_total_counts['counts'].max() + 10

    # create a figure
    plt.figure(figsize=(10, 6))

    # ensure 'gsd_m' and 'native_resolution' are of the same type
    native_res_value = native_res.loc[native_res['img_id'] == img_cat_id, 'native_resolution_m'].values
    native_res_value = native_res_value.item() # convert array to scalar

    # filter the data for native resolution
    native_res_df = filtered_sat_total_counts[filtered_sat_total_counts['gsd_m'] == native_res_value].copy() # use .copy() to avoid SettingWithCopyWarning

    # for each img_cat_id plot the filtered total count per observer for each gsd_m with pointplot
    ax = sns.pointplot(
        x = 'gsd_m',
        y = 'counts',
        hue = 'observer', # colours based on number of observers
        markers = ['s', 'o', '^'], # define the marker style for each observer
        linestyles = ['-', '--', '-.'], # define the line plotting style for each observer
        palette = 'colorblind',
        data = filtered_sat_total_counts,
        order = sorted(filtered_sat_total_counts['gsd_m'].unique(), reverse = True) # include only present gsd_m values in descending order
    )

    # capture x-axis ticks and limits from the dataset
    x_ticks = ax.get_xticks() # get the x-ticks from the dataset
    x_limits = ax.get_xlim() # get the x-axis limits

    # set all markers to no fill (outline only)
    for line in ax.lines:
        line.set_markerfacecolor('none') # set no fill for the marker
        line.set_markeredgecolor(line.get_color()) # set edge colour to match line colour

    # create a new dataframe for the plot to overlay filtered total counts associated with native resolution as filled points, so they are easily identifiable
    # ensure to include all gsd_m values
    native_plotting_data = filtered_sat_total_counts.copy()
    native_plotting_data['counts'] = native_plotting_data['counts'].where(native_plotting_data['gsd_m'] == native_res_value, float('nan'))

    # overlay the modified data (native resolution + NaN counts for others) using pointplot with no lines
    sns.pointplot(
        x = 'gsd_m',
        y = 'counts',
        hue = 'observer', # different colour based on observer
        data = native_plotting_data, # modified data with NaN counts for non-native resolutions
        markers = ['s', 'o', '^'], # assign markers for each observer
        palette = 'colorblind',
        linestyles = ['','',''], # no lines between points
        ax = ax,
        dodge = False, # disable horizontal adjustment to avoid overlap of points
        legend = True # enable the legend for scatterplot to add to legend
    )

    # set x-ticks and x-limits to ensure alignment
    ax.set_xticks(x_ticks) # reapply the x-ticks from the original pointplot
    ax.set_xlim(x_limits) # lock the x-axis limits to match the dataset

    # determine the y-axis limit by which is larger, the total count max or ground reference count
    if img_cat_id in ref_satellite_list:
        ref_value = ref_satellite_list[img_cat_id]
        y_max = max(axis_max, ref_value + 10)
        ax.axhline(ref_value, color='0.5', linestyle='--', label=f'Reference: {ref_value}')
        ax.axhspan(0, ref_value, color='0.7', alpha=0.3)
    else:
        y_max = axis_max

    # combine legend handles and labels manually after plotting scatterplot
    handles, labels = ax.get_legend_handles_labels()

    # print the handles to understand what is being generated is correct
    print("Handles before filtering:", handles)
    print("Labels before filtering:", labels)

    # condition to check if there is only the native resolution
    unique_gsd_m = filtered_sat_total_counts['gsd_m'].nunique()

    # get unique observers for the current image_cat_id to dynamically adjust labels
    available_observers = sorted(filtered_sat_total_counts['observer'].unique())
    unique_gsd_m = filtered_sat_total_counts['gsd_m'].nunique()

    # modify handles and labels based on conditions
    if unique_gsd_m == 1:  # only the native resolution is present
        # only include the second native point plot handles, exclude the first set which is for the original line plot
        observer_count = len(available_observers)

        # assuming the first 'observer_count' handles correspond to the line plot
        handles = handles[observer_count:]

        # dynamically create labels based on available observers at native resolution
        labels = [f'Observer {obs}: Native Resolution' for obs in available_observers]

        # include the reference label if available
        if img_cat_id in ref_satellite_list:
            labels.append(f'Reference: {ref_value}')
    else:
        # multiple resolutions, include both line plot and native resolution point plot in the legend
        labels = [f'Observer {obs}' for obs in available_observers] + \
                 [f'Observer {obs}: Native Resolution' for obs in available_observers]

        # add reference label if available
        if img_id in ref_satellite_list:
            labels.append(f'Reference: {ref_value}')

    # print the updated handles and labels to verify
    print("Handles after filtering:", handles)
    print("Labels after filtering:", labels)

    # call ax.legend() after modifying handles and labels
    ax.legend(handles=handles[:len(labels)], labels=labels, bbox_to_anchor=(1.05, 1), loc='upper left')

    # set the axis limit and amend the axis names
    ax.set(ylim=(0, y_max))
    plt.xlabel('Image Resolution (m)')
    plt.ylabel('Total Count: Definite and Likely (No. of Individuals)')
    plt.title(f'Total count: {img_filename}')

    # update the x-axis labels to bold and asterisk the gsd_m which is the native resolution for the particular sensor
    x_labels = ax.get_xticklabels()
    new_labels = []
    for label in x_labels:
        text = label.get_text()
        if text and float(text) == native_res_value:
            new_labels.append(r'$\mathbf{' + text + r'^*}$')
        else:
            new_labels.append(text)
    ax.set_xticklabels(new_labels)

    # save the plot
    file_path = os.path.join(output_path, f'{img_filename}_filtered_total_count.png')
    print(f"Saving plot to: {file_path}")
    plt.savefig(file_path, bbox_inches='tight')
    # plt.savefig(os.path.join(output_path, f'Plot for image:{img_filename}.png'), dpi=100)

    # show the plot
    plt.show()

    # close the figure to avoid memory issues
    plt.close()

For each image catalogue id, for each observer, get a breadown of total count by certainty, for each spatial resolution:

In [ ]:
# for each image catalogue id, get a breadown of total count by certainty for each observer, at each spatial resolution
# https://datagy.io/pandas-count-unique-values-groupby/
# create a dictionary to store the resulting dataframes
sat_certainty_counts_dfs = {}

# loop over concatenated_by_img and group data by certainty for each observer within each gsd_m group, to get the count per certainty
for img_cat_id, df in concatenated_by_img.items():
    print(f"Unique certainty value counts for img_cat_id '{img_cat_id}':")
    sat_certainty_counts = df.groupby(['gsd_m', 'observer'])['certainty'].value_counts() # get the total count per certainty
    sat_certainty_counts_df = sat_certainty_counts.reset_index(name = 'counts') # reset the column name for certainty counts to counts

    # store the dataframe sat_certainty_counts_df in the dictionary
    sat_certainty_counts_df_pivot = sat_certainty_counts_df.pivot_table(index = ['gsd_m', 'observer'], columns = 'certainty', values = 'counts', fill_value = 0).reset_index()
    sat_certainty_counts_df_pivot.set_index(['gsd_m', 'observer'], inplace=True)
    print(sat_certainty_counts_df_pivot)
    sat_certainty_counts_dfs[img_cat_id] = sat_certainty_counts_df_pivot

# for img_id, pivot_df in certainty_counts_dfs.items():

In [ ]:
# to make bar plots of a breadown of total count by certainty for each observer, at each spatial resolution, assign each observer certainty a unique colour

# assign colours to each observer by certainty level
# for lik ('likely_70-89') and pos ('possible_50-69'), colours are defined as a lighter shade of the colourbline friendly colours selected for def ('definite_90-100') values
observer_colors = {
    '1def': colorblind_palette['blue'],
    '2def': colorblind_palette['orange'],
    '3def': colorblind_palette['blueish_green'],
    '1lik': (0.0, 0.65, 0.85),
    '2lik': (0.93, 0.80, 0.20),
    '3lik': (0.2, 0.76, 0.65),
    '1pos': (0.2, 0.75, 0.95),
    '2pos': (0.95, 0.90, 0.40),
    '3pos': (0.4, 0.86, 0.75)
}

In [ ]:
# plot a breakdown of total count by certainty for each observer, at each spatial resolution

# unstack the 'observer' column
for img_cat_id, sat_certainty_counts_df_pivot in sat_certainty_counts_dfs.items():
    # Unstack to separate observers as columns
    df_unstacked = sat_certainty_counts_df_pivot.unstack(level=-1)

    # define stacking order (reversed to make "definite" at the bottom)
    stacking_order = ['definite_90-100', 'likely_70-89', 'possible_50-69']

    # get native resolution for the current image catalogue ID
    native_res_value = native_res.loc[native_res['img_id'] == img_cat_id, 'native_resolution_m'].values.item()

    # create a filtered dataframe for native resolution only
    df_native = df_unstacked.copy()
    df_native[df_native.index != native_res_value] = None  # Set all non-native resolution values to NaN

    # initialise figure for the stacked bar plot
    fig, ax = plt.subplots(figsize=(14, 6))  # increased figure width further to make space for plot and legend

    # add reference line and shaded region if reference exists for the current img_cat_id
    reference_line = None
    ref_value = None
    if img_cat_id in ref_satellite_list:
        ref_value = ref_satellite_list[img_cat_id]
        # add the shaded region first with lower z-order
        ax.axhspan(0, ref_value, color='0.7', alpha=0.3, zorder=1, label='_nolegend_')
        # add the reference line
        reference_line = ax.axhline(ref_value, color='0.5', linestyle='--', zorder=2, label=f'Reference: {ref_value}')

    # plot bars for each observer
    bar_width = 0.25  # width of each observer's bars
    x = range(len(df_unstacked))  # positions for bars (index of the dataframe)
    offsets = [-bar_width, 0, bar_width]  # offset positions for observers

    # initialise list for legend handles and labels
    legend_handles = []
    legend_labels = []

    # initialise empty lists to store the correct handles for the categories
    observer_handles = {'1': [], '2': [], '3': []}

    for i, observer in enumerate(['1', '2', '3']):
        observer_df = df_unstacked.xs(observer, level='observer', axis=1)  # filter by observer

        observer_colors_list = [
            observer_colors[f'{observer}def'],  # colour for 'definite'
            observer_colors[f'{observer}lik'],  # colour for 'likely'
            observer_colors[f'{observer}pos'],  # colour for 'possible'
        ]

        # stack bars for the current observer
        bottom_stack = 0  # reset for each observer
        for j, category in enumerate(stacking_order):  # iterate through categories (definite, likely, possible)
            if category in observer_df.columns:
                bars = ax.bar(
                    [p + offsets[i] for p in x],  # adjust x positions for each observer
                    observer_df[category],  # heights of the bars
                    bar_width,  # width of each bar
                    color=observer_colors_list[j],  # color for this observer/category
                    bottom=bottom_stack,  # stack bars correctly
                    edgecolor='black',  # add edgecolor to create the hollow effect
                    linewidth=1  # add a border width for clarity
                )
                # add the bar handle to the list for later use in the legend (only once for each category)
                observer_handles[observer].append(bars[0])  # append the first bar of each category for legend

                # update the bottom stack for the next category
                bottom_stack += observer_df[category]

    # now, construct the legend handles for each observer in the correct order
    for observer in ['1', '2', '3']:
        for j, category in enumerate(stacking_order):
            if category in df_unstacked.columns:
                legend_labels.append(f'Observer {observer} {category.capitalize()}')
                legend_handles.append(observer_handles[observer][j])  # use pre-collected handles

    # add the native resolution black line handle to the legend
    if not df_native[df_native.index == native_res_value].empty:
        native_resolution_line = Line2D([0], [0], color='black', lw=1)  # create black line handle
        legend_handles.append(native_resolution_line)
        legend_labels.append('Native Resolution')

    # add the reference line and shading if exists
    if reference_line is not None:
        legend_handles.append(reference_line)
        legend_labels.append(f'Reference: {ref_value}')

    # set the final legend with proper labels and handles, positioned outside the plot area
    ax.legend(legend_handles, legend_labels, bbox_to_anchor=(1.05, 1), loc='upper left', ncol=1, fontsize=10)  # 1 column for the legend

    # get the y-axis limits after plotting
    y_min, y_max = ax.get_ylim()

    # calculate the y-axis limit
    y_limit = max(y_max + 5, ref_value + 5 if img_cat_id in ref_satellite_list else y_max + 5)
    ax.set_ylim(0, y_limit)

    # set x-ticks to show actual resolutions
    ax.set_xticks(x)  # set x-ticks to correspond to the index of `df_unstacked`
    ax.set_xticklabels(df_unstacked.index)  # set x-tick labels to the resolutions

    # reverse x-axis to invert the order of resolutions (if you still want that)
    ax.invert_xaxis()

    # set the axis labels and title (align title to left)
    ax.set_xlabel('Image Resolution (m)')
    ax.set_ylabel('Total Count (No. of Individuals)')
    # set the axis title, center-aligned to the entire figure area (including space for the legend)
    ax.set_title(f'Image Catalogue ID: {img_cat_id}', loc='center')  # title centered on the entire figure area

    # remove gridlines and ensure layout fits neatly
    ax.grid(False)

    # use tight_layout with extra padding to give space for the legend
    plt.tight_layout(pad=2.5)

    # save the plot
    file_path = os.path.join(output_path, f'Plot_for_img_id_{img_cat_id}_certainty.png')
    print(f"Saving plot to: {file_path}")
    plt.savefig(file_path, bbox_inches='tight')

    # show plot
    plt.show()

    # close the figure to avoid memory issues
    plt.close()

Code to replicate plotting less of title and legend:

In [ ]:
# plot a breadown of total count by certainty for each observer, at each spatial resolution

# unstack the 'observer' column
for img_cat_id, sat_certainty_counts_df_pivot in sat_certainty_counts_dfs.items():
    # unstack to separate observers as columns
    df_unstacked = sat_certainty_counts_df_pivot.unstack(level=-1)  # unstack the 'observer' column

    # define stacking order (reversed to make "definite" at the bottom)
    stacking_order = ['definite_90-100', 'likely_70-89', 'possible_50-69']

    # get native resolution for the current image catalogue ID
    native_res_value = native_res.loc[native_res['img_id'] == img_cat_id, 'native_resolution_m'].values.item()

    # create a filtered dataframe for native resolution only
    df_native = df_unstacked.copy()
    df_native[df_native.index != native_res_value] = None  # set all non-native resolution values to NaN

    # reset all settings before applying changes
    plt.rcParams.update(plt.rcParamsDefault)
    plt.rcParams['font.size'] = 18  # Set default font size
    plt.rcParams['font.family'] = 'Arial'

    # initialise figure for the stacked bar plot
    fig, ax = plt.subplots(figsize=(10, 6))

    # add reference line and shaded region if reference exists for the current img_cat_id
    reference_line = None
    if img_cat_id in ref_satellite_list:
        ref_value = ref_satellite_list[img_cat_id]
        # add the shaded region first with lower z-order
        ax.axhspan(0, ref_value, color='0.7', alpha=0.3, zorder=1, label='_nolegend_')
        # add the reference line
        reference_line = ax.axhline(ref_value, color='0.5', linestyle='--', zorder=2, label=f'Reference: {ref_value}')

    # plot bars for each observer (as usual, full bars)
    bar_width = 0.25  # width of each observer's bars
    x = range(len(df_unstacked))  # positions for bars (index of the dataframe)
    offsets = [-bar_width, 0, bar_width]  # offset positions for observers

    # plot the full bars (stacked) for each observer independently
    bottom_stack = {observer: 0 for observer in ['1', '2', '3']}  # reset bottom for each observer

    for i, observer in enumerate(['1', '2', '3']):
        observer_df = df_unstacked.xs(observer, level='observer', axis=1)  # filter by observer

        observer_colors_list = [
            observer_colors[f'{observer}def'],  # colour for 'definite'
            observer_colors[f'{observer}lik'],  # colour for 'likely'
            observer_colors[f'{observer}pos'],  # colour for 'possible'
        ]

        # stack bars for the current observer
        for j, category in enumerate(stacking_order):  # iterate through categories (definite, likely, possible)
            if category in observer_df.columns:
                ax.bar(
                    [p + offsets[i] for p in x],  # adjusted x positions for each observer
                    observer_df[category],  # heights of the bars
                    bar_width,  # width of each bar
                    color=observer_colors_list[j],  # colour for this observer/category
                    bottom=bottom_stack[observer],  # stack bars correctly for each observer independently
                )
                # update bottom for the next category for this observer
                bottom_stack[observer] += observer_df[category]

    # overlay the hollow bars for the native resolution
    if not df_native[df_native.index == native_res_value].empty:
        native_bottom_stack = {observer: 0 for observer in ['1', '2', '3']}  # reset bottom stack for native resolution

        for i, observer in enumerate(['1', '2', '3']):
            observer_df = df_native.xs(observer, level='observer', axis=1)  # filter by observer

            observer_colors_list = [
                observer_colors[f'{observer}def'],  # colour for 'definite'
                observer_colors[f'{observer}lik'],  # colour for 'likely'
                observer_colors[f'{observer}pos'],  # colour for 'possible'
            ]

            for j, category in enumerate(stacking_order):  # iterate through categories (definite, likely, possible)
                if category in observer_df.columns:
                    ax.bar(
                        [p + offsets[i] for p in x],  # adjusted x positions for each observer
                        observer_df[category],  # heights of the bars
                        bar_width,  # width of each bar
                        color='none',  # no fill for native resolution bars (hollow)
                        edgecolor='black',  # black outline for native resolution bars
                        linewidth=2,  # increase the weight of the line (e.g., 2 for thicker lines)
                        bottom=native_bottom_stack[observer],  # stack bars correctly for native resolution
                    )
                    native_bottom_stack[observer] += observer_df[category]  # update bottom for native resolution

    # get the y-axis limits after plotting
    y_min, y_max = ax.get_ylim()

    # calculate the y-axis limit
    y_limit = max(y_max + 5, ref_value + 5 if img_cat_id in ref_satellite_list else y_max + 5)
    ax.set_ylim(0, y_limit)

    # set x-ticks to show actual resolutions
    ax.set_xticks(x)  # set x-ticks to correspond to the index of `df_unstacked`
    ax.set_xticklabels(df_unstacked.index)  # set x-tick labels to the resolutions

    # reverse x-axis to invert the order of resolutions (if you still want that)
    ax.invert_xaxis()

    # set the axis labels and title
    ax.set_xlabel('Image Resolution (m)')
    ax.set_ylabel('Total Count (No. of Individuals)')
    # ax.set_title(f'Image Catalogue ID: {img_cat_id}')

    # remove gridlines and ensure layout fits neatly
    ax.grid(False)
    plt.tight_layout()

    # save the plot
    file_path = os.path.join(output_path, f'Plot_for_img_id_{img_cat_id}_certainty.png')
    print(f"Saving plot to: {file_path}")
    plt.savefig(file_path, bbox_inches='tight')
    # plt.savefig(os.path.join(output_path, f'Plot for img_id:{img_id}.png'), dpi=100)

    # show plot
    plt.show()

    # close the figure to avoid memory issues
    plt.close()

The following code can be unhashed to filter the dataframes to remove any possible detections, enabling users to perform clustering using only those detections with high certainty.

In [ ]:
# # create a new dictionary to store the filtered dataframes
# filter_poss_concatenated_csv_dfs = {}

# # iterate through each group and dataframe in the original concatenated dictionary
# for group, df in concatenated_csv_dfs.items():
#     # Apply the filtering condition
#     filtered_df = df[df['certainty'] != 'possible_50-70'].reset_index(drop=True)

#     # store the filtered dataframe in the new dictionary
#     filter_poss_concatenated_csv_dfs[group] = filtered_df

#     # print the filtered dataframe to verify
#     print(f"Filtered DataFrame for group '{group}':")
#     print(filter_poss_concatenated_csv_dfs[group])

### Satellite clustering analysis

Prepare the satellite data for clustering: <br>

Create functions necessary to work with raster data:

In [ ]:
# get_raster_filename gets the path of the raster and extracts the image filename
def get_raster_filename(rstr_path):
    '''
    Function to extract the raster filename from the file path.
    '''
    path, filename = os.path.split(rstr_path)
    return filename

# get_raster_sensor gets the path of the raster and extracts the image filename and store the value corresponding to sensor in the filename
# sensor can be extract by defining a seperator as _ and the location of the value in this instance [0], remember python uses 0 indexing/values start from 0
# this function expects files to be named with the following naming convention: 'location_satellite-sensor_image-id_image-date_image-time_resolution_observer-id.csv
def get_raster_sensor(rstr_path):
    '''
    Function to extract the raster sensor from the filename.
    '''
    path, filename = os.path.split(rstr_path)
    filename, exe = os.path.splitext(filename)
    indices = [1]
    separator = '_'
    parts = filename.split(separator)
    return separator.join([parts[i] for i in indices])

# open_raster uses rasterio to open the image
def open_raster(rstr_path):
    '''
    Function to open a raster using rasterio.
    '''
    img = rasterio.open(rstr_path)
    return img

# get_raster_as_array uses rasterio to open an image and read in as an array
def get_raster_as_array(rstr_path):
    '''
    Function to open a raster using rasterio as an array.
    '''
    with rasterio.open(rstr_path) as arr:
        return arr.read()

# get_raster_band_count uses rasterio to open the image and extracts the number of bands in the image
# for example, for Vantor optical satellite imagery you might expect 4 or 8 band imagery
# for TerraSAR-X a sythetic apeture radar (SAR) imagery, you can expect 1 band imagery
def get_raster_band_count(rstr_path):
    '''
    Function to extract the number of bands of an image from the image metadata.
    '''
    with rasterio.open(rstr_path) as img:
        return img.count

# get_crs uses rasterio to open an image and extract the image coordinate reference system (CRS)
def get_crs(rstr_path):
    '''
    Function to extract the image coordinate reference system (CRS) from the image metadata.
    '''
    with rasterio.open(rstr_path) as dataset:
        return dataset.crs

# convert_coordinates transforms geographical coordinates from one coordinate reference system (CRS) to another
# in this instance it converts the point coordinates (likely in espg:4326 if following the 'Cetacean Strandings from Space' pipeline) to image crs
# the function creates a Transformer object using the source CRS (src_crs) and destination CRS (dst_crs)
# always_xy=True parameter ensures that the coordinates are always interpreted as (longitude, latitude)
def convert_coordinates(df, src_crs, dst_crs):
    '''
    Function to transform geographical coordinates from one coordinate reference system (CRS) to another.
    '''
    transformer = Transformer.from_crs(src_crs, dst_crs, always_xy=True)
    # apply the transformation to each row of the df, converting the longitude and latitude of each row from the src_crs to the dst_crs
    # store the transformed coordinates as converted_lon and converted_lat
    df['converted_lon'], df['converted_lat'] = zip(*df.apply(lambda row: transformer.transform(row['longitude'], row['latitude']), axis=1))
    return df

# latlon_to_pixel converts geographical coordinates (latitude and longitude) to pixel coordinates based on a given transformation matrix
def latlon_to_pixel(lat, lon, transform):
    '''
    Function to convert geographical coordinates (latitude and longitude) to pixel coordinates based on a given transformation matrix.
    '''
    # ~transform part computes the inverse of the transformation matrix
    # multiply the inverse transformation matrix by the (longitude, latitude) tuple to convert the geographical coordinates into pixel coordinates
    row, col = ~transform * (lon, lat)
    # pixel coordinates are returned as integers
    return int(row), int(col)

# make_plotting_array processes satellite imagery data to create an RGB image array suitable for plotting
def make_plotting_array(full_arr, sensor_bands, sensor_name, percent_clip=2):
    '''
    Function to process satellite imagery data to create an RGB image array suitable for plotting.
    Extract the bands from full image array, for the correct satellite sensor apply their known band configurations for RGB to create an rgb array.
    Else, if the satellite sensor is a SAR sensor tsx-1 extract the 1 band as an array.
    For optical sensors:
        Calculate the percentiles p1 and p2 for the rgb_arr based on the percent_clip value to clip the intensity values.
        Rescale the intensity of rgb_arr to the range [0, 1] using the calculated percentiles.
        Transpose the rescaled array to the shape (height, width, channels) and return.
    Else, return None for the SAR image.
    '''
    # sensor_band_mapping defines a dictionary that maps different sensor names and their known band configurations to specific RGB channels
    sensor_band_mapping = {
        'geoeye1': {4: [2, 1, 0], 8: [4, 2, 1]},
        'wv2': {4: [2, 1, 0], 8: [4, 2, 1]},
        'wv3': {4: [2, 1, 0], 8: [4, 2, 1]},
        'pleiades': {4: [2, 1, 0], 8: [3, 2, 1]},
        'pleiades-neo': {4: [2, 1, 0], 8: [3, 2, 1]},
        'skysat': {4: [2, 1, 0], 8: [3, 2, 1]},
        'pleiades-sentinel-hub': {4: [0, 1, 2]}
    }
    # if sensor_name and sensor_bands from the image name are in the sensor_band_mapping
    if sensor_name in sensor_band_mapping and sensor_bands in sensor_band_mapping[sensor_name]:
        # extract the corresponding bands from full_arr to create an rgb_arr
        rgb_arr = np.array([full_arr[i] for i in sensor_band_mapping[sensor_name][sensor_bands]])
    elif sensor_name == 'tsx-1' and sensor_bands == 1:
        # if the sensor is 'tsx-1' (a SAR sensor) with sensor_bands equal to 1, return the first band as a SAR image
        sar_image = full_arr[0]
        return None, sar_image
    else:
        # if the sensor and bands do not match the expected values, it prints an error message and returns None
        print(f"Expected sensor to be either '4 band' or '8 band', input {sensor_bands}")
        return None, None

    # calculate the percentiles p1 and p2 for the rgb_arr based on the percent_clip value to clip the intensity values
    p1, p2 = np.percentile(rgb_arr, (percent_clip, 100 - percent_clip))
    # rescale the intensity of rgb_arr to the range [0, 1] using the calculated percentiles
    rgb_rescale = exposure.rescale_intensity(rgb_arr, in_range=(p1, p2), out_range=(0, 1))
    # transpose the rescaled array to the shape (height, width, channels) and return for optical and return None for the SAR image
    return rgb_rescale.transpose(1, 2, 0), None

Group the satellite imagery by spatial resolution and image catalogue ID to match with the corresponding count .csv data.

In [ ]:
# access each .tif file path, split the path to extract the image filename and store in a list called tif_img_name
tif_img_name = []
for img in input_satellite_img_tif:
    dir_img, file_img = os.path.split(img)
    tif_img_name.append(file_img)
print(tif_img_name)

# define a function to group satellite image filenames by separator indices 5 ('gsd_m') and 2 ('img_cat_id')
# this function expects files to be named with the following naming convention: 'location_satellite-sensor_image-id_image-date_image-time_resolution_observer-id.csv
# Function `group_by_indices_exc_observer` is imported from strandings_from_space.filenames.
# group the .tif filenames by spatial resolution and image id
print('All .tif files: group and value')
# group the strings
grouped_tif_name = group_by_indices_exc_observer(tif_img_name, separator='_', indices=[5, 2])
print(grouped_tif_name)

# convert groups 'gsd_m' and 'img_cat_id' to a list of keys
print('All .tif files: group')
# convert to a list of keys
tif_group_keys = list(grouped_tif_name.keys())
print(tif_group_keys)

Plot all observers data points on the corresponding satellite image.

In [ ]:
# define marker style based on tool_label
marker_map = {'1':'s', '2': 'o', '3': '^'}

plt.rcParams.update(plt.rcParamsDefault)

# loop through concatenated dataframes and if the dataframe and matched image exist, open the image with rasterio
for (group_1, df), (group_2, tif) in zip(concatenated_csv_dfs.items(), grouped_tif_name.items()):
    if group_1 == group_2:
        tif_file = os.path.join(input_satellite_images, tif)
        with rasterio.open(tif_file) as dataset:
            output_name = tif[:-4]

            # extract CRS information
            dst_crs = get_crs(tif_file)
            src_crs = CRS.from_epsg(4326)

            # convert coordinates
            dataframe = convert_coordinates(df, src_crs, dst_crs)

            # extract geolocation information
            full_arr = get_raster_as_array(tif_file)
            sensor_name = get_raster_sensor(tif_file)
            print(f'sensor_name: {sensor_name}')
            sensor_bands = get_raster_band_count(tif_file)
            print(f'sensor_bands: {sensor_bands}')
            transform = dataset.transform
            origin_x = transform.c
            pixel_width = transform.a
            rotation_x = transform.b
            origin_y = transform.f
            rotation_y = transform.d
            pixel_height = transform.e
            print(f'image size is, {origin_x, pixel_width, rotation_x, origin_y, rotation_y, pixel_height}')
            width = dataset.width
            height = dataset.height
            print(f'Image width: {width}, Image height: {height}')

            # convert georeferenced coordinates (latitude, longitude) into pixel-based coordinates within the image (row, col), using an affine transform
            dataframe['pixel_x'], dataframe['pixel_y'] = zip(*dataframe.apply(lambda row: latlon_to_pixel(row['converted_lat'], row['converted_lon'], transform), axis=1))

            print(dataframe.head())

            # stretch the intensity level of the image for visualising
            plot_rescaled, sar_image = make_plotting_array(full_arr, sensor_bands, sensor_name, percent_clip=2)
            if plot_rescaled is not None:
                plt.imshow(plot_rescaled)
                plt.title(f'{output_name}')

                # assign unique colours to observers
                observer_colors = {
                    observer: colorblind_palette[color] for observer, color in zip(unique_observer_ids, ['blue', 'orange', 'blueish_green'])
                }
                print(observer_colors)
                # plot data points for each row in the subset dataframe
                for _, row in df.iterrows():
                    x = row['pixel_x']
                    y = row['pixel_y']
                    label = row['observer']
                    user_color = observer_colors[row['observer']]
                    marker = marker_map.get(label, 'o')  # Default to circle if label is not found
                    # Plot points
                    plt.scatter(x, y, color=user_color, marker=marker, edgecolor='black', s=25)  # Adjust s for size

                # remove gridlines
                plt.grid(False)

                # save the annotated image
                output_img_path = os.path.join(output_path, f'{output_name}_total_counts.png')
                plt.savefig(output_img_path, bbox_inches='tight')

                # show the plot inline
                plt.show()
                print(f"Annotation complete for {output_name}. Saved in '{output_path}' folder.")

                plt.close()  # close the figure

            elif sar_image is not None:
                plt.figure(figsize=(10, 10))
                plt.imshow(sar_image, cmap='gray', vmin=0, vmax=988) # adjust vmin and vmax accordingly
                plt.title(f'{output_name}')
                plt.colorbar()

                # assign unique colours to observers
                observer_colors = {
                    observer: colorblind_palette[color] for observer, color in zip(unique_observer_ids, ['blue', 'orange', 'blueish_green'])
                }
                print(observer_colors)
                # plot data points for each row in the subset dataframe
                for _, row in df.iterrows():
                    x = row['pixel_x']
                    y = row['pixel_y']
                    label = row['observer']
                    user_color = observer_colors[row['observer']]
                    marker = marker_map.get(label, 'o')  # default to circle if label is not found
                    # Plot points
                    plt.scatter(x, y, color=user_color, marker=marker, edgecolor='black', s=25)  # adjust s for marker size

                # remove gridlines
                plt.grid(False)

                # save the annotated image
                output_img_path = os.path.join(output_path, f'{output_name}_total_counts.png')
                plt.savefig(output_img_path, bbox_inches='tight')

                plt.show()
                print(f"Annotation complete for {output_name}. Saved in '{output_path}' folder.")

                plt.close()  # close the figure

Wards's based clustering: <br>
Define the parameters for clustering (pixel based distance), to initially cluster using Ward's method and then to split cluster by constraits whereby no cluster can contain an observer more than once, or more than the maximum number of unique observers.

In [ ]:
def split_clusters_by_constraints(data, distance_threshold_pixels, max_labels_per_cluster=3):
    '''
    Function to split clusters further by constraints. 
    No cluster should exceed three points and does not contain the same observer more than once.
    Else remove the furthest instance of an observer and reassign to the nearest cluster that meets these parameters else assign to a new cluster.
    '''
    new_cluster_id = data['cluster'].max() + 1  # start new cluster IDs after the existing ones
    print(f'new cluster id is {new_cluster_id}')

    for cluster_id in data['cluster'].unique():
        cluster_data = data[data['cluster'] == cluster_id]

        # calculate the centroid of the cluster
        centroid_x = cluster_data['pixel_x'].mean()
        centroid_y = cluster_data['pixel_y'].mean()

        # ensure each observer is in a separate cluster within the same cluster group
        for observer, user_group in cluster_data.groupby('observer'):
            if len(user_group) > 1:
                # calculate the distance of each instance from the centroid
                user_group['distance'] = np.sqrt((user_group['pixel_x'] - centroid_x)**2 + (user_group['pixel_y'] - centroid_y)**2)

                # sort by distance and keep the closest instance
                user_group = user_group.sort_values(by='distance')
                closest_instance_idx = user_group.index[0]

                # reassign all other instances to new clusters
                for idx in user_group.index[1:]:
                    point_x, point_y = user_group.loc[idx, ['pixel_x', 'pixel_y']]

                    # search for nearby clusters within the distance threshold
                    nearby_clusters = data[(data['cluster'] != cluster_id) &  # exclude current cluster
                                           (np.sqrt((data['pixel_x'] - point_x) ** 2 + (data['pixel_y'] - point_y) ** 2) < distance_threshold_pixels) &  # Within distance threshold
                                           (data['observer'] != observer)]  # exclude clusters with the same observer

                    reclustered = False
                    for nearby_cluster_id in nearby_clusters['cluster'].unique():
                        nearby_cluster_data = data[data['cluster'] == nearby_cluster_id]
                        label_count = len(nearby_cluster_data)

                        # check if the cluster has less than 3 labels and doesn't already contain the same observer
                        if label_count < 3 and observer not in nearby_cluster_data['observer'].values:
                            # reassign point to this nearby cluster
                            data.loc[idx, 'cluster'] = nearby_cluster_id
                            reclustered = True
                            break  # no need to check further once reassigned

                    if not reclustered:
                        # if no valid nearby cluster found, create a new cluster
                        data.loc[idx, 'cluster'] = new_cluster_id
                        new_cluster_id += 1

    return data

# function to perform clustering with distance threshold
def perform_clustering(data, distance_threshold, max_labels_per_cluster=3):
    '''
    Function to perform hierarchical clustering using Wards method, within a pixel based distance threshold to a maximum of three points per cluster.
    '''
    if data.empty:
        raise ValueError("Data is empty")

    # prepare the coordinates for clustering
    coords = data[['pixel_x', 'pixel_y']].values

    # perform hierarchical clustering (Ward method)
    linkage_matrix = linkage(coords, method='ward')

    # apply clustering with the pixel-based distance threshold
    clusters = fcluster(linkage_matrix, distance_threshold, criterion='distance')

    # add cluster results to the data
    data['cluster'] = clusters

    # split clusters based on constraints
    data = split_clusters_by_constraints(data, distance_threshold, max_labels_per_cluster = max_labels_per_cluster)

    return data

In [ ]:
# the distance threshold meters and resulting distance threshold in pixels for each satellite image in the experiments in:
# Clarke et al., (2026): 'Odontocete strandings from space: Accurately counting individuals with very high-resolution optical and synthetic aperture radar satellite imagery'
# are detailed below:

# # '('28', '3490335')': distance_threshold_meters = 1.5, meters_per_pixel = 0.2741195689620352
# print(f'distance threshold pixels for image cat ID 28, 3490335 = {1.5 / 0.2741195689620352}')
# # '('30', '104001008D07E000')': distance_threshold_meters = 1.5, meters_per_pixel = 0.3
# print(f'distance threshold pixels for 30, 104001008D07E000 = {1.5 / 0.3}')
# # '('50', '105001002F60AA00')': distance_threshold_meters = 2.25, meters_per_pixel = 0.5
# print(f'distance threshold pixels for 50, 105001002F60AA00 = {2.25 / 0.5}')
# # '('30', '10300100DC306300')': distance_threshold_meters = 2.2, meters_per_pixel = 0.3
# print(f'distance threshold pixels for 30, 10300100DC306300 = {2.2 / 0.3}')
# # '('50', '10300100DC306300')': distance_threshold_meters = 2.54, meters_per_pixel = 0.5
# print(f'distance threshold pixels for 50, 10300100DC306300 = {2.54 / 0.5}')
# # '('15', '104001007E5E7400')': distance_threshold_meters = 1.5, meters_per_pixel = 0.15
# print(f'distance threshold pixels for 15, 104001007E5E7400 = {1.5 / 0.15}')
# # '('30', '104001007E5E7400')': distance_threshold_meters = 1.5, meters_per_pixel = 0.3
# print(f'distance threshold pixels for 30, 104001007E5E7400 = {1.5 / 0.3}')
# # '('50', '104001007E5E7400')': distance_threshold_meters = 2.7, meters_per_pixel = 0.5
# print(f'distance threshold pixels for 50, 104001007E5E7400 = {2.7 / 0.5}')
# # '('30', '105001002F0CA400')': distance_threshold_meters = 1.8, meters_per_pixel = 0.3
# print(f'distance threshold pixels for 30, 105001002F0CA400 = {1.8 / 0.3}')
# # '('50', '105001002F0CA400')': distance_threshold_meters = 1.8, meters_per_pixel = 0.5
# print(f'distance threshold pixels for 50, 105001002F0CA400 = {1.8 / 0.5}')
# # '('50', '10300100DB012A00')': distance_threshold_meters = 2.25, meters_per_pixel = 0.5
# print(f'distance threshold pixels for 50, 10300100DB012A00 = {2.25 / 0.5}')
# # '('50', 'DS-PHR1B-201811282257268-FR1-PX-E167S47-0909-00925')': distance_threshold_meters = 1.8, meters_per_pixel = 0.5
# print(f'distance threshold pixels for 50, DS-PHR1B-201811282257268-FR1-PX-E167S47-0909-00925 = {1.8 / 0.5}')
# # '('30', '1030010089B22D00')': distance_threshold_meters = 1.8, meters_per_pixel = 0.3
# print(f'distance threshold pixels for 30, 1030010089B22D00 = {1.8 / 0.3}')
# # '('50', '1030010089B22D00')': distance_threshold_meters = 2.0, meters_per_pixel = 0.5
# print(f'distance threshold pixels for 50, 1030010089B22D00 = {2.0 / 0.5}')

In [ ]:
# define the mapping of (gsd, image_id) to distance threshold values
# else if distance_threshold_meters value is applicable to all imagery, this is not necessary and distance_threshold_meters can be defined in the next code cell
distance_thresholds = {
    ('28', '3490335'): 1.5,
    ('30', '104001008D07E000'): 1.5,
    ('50', '105001002F60AA00'): 2.25,
    ('30', '10300100DC306300'): 2.2,
    ('50', '10300100DC306300'): 2.54,
    ('15', '104001007E5E7400'): 1.5,
    ('30', '104001007E5E7400'): 1.5,
    ('50', '104001007E5E7400'): 2.7,
    ('30', '105001002F0CA400'): 1.8,
    ('50', '105001002F0CA400'): 1.8,
    ('50', '10300100DB012A00'): 2.25,
    ('50', 'DS-PHR1B-201811282257268-FR1-PX-E167S47-0909-00925'): 1.8,
    ('30', '1030010089B22D00'): 1.8,
    ('50', '1030010089B22D00'): 2.0,
}

The following code clusters annotations across all certainty levels. <br> 

To perform clustering on filtered data less of low certainty detections assigned 'possible_50-69', amend concatenated_csv_df for filter_poss_concatenated_csv_dfs in for (group_1, df), (group_2, tif) in zip(concatenated_csv_dfs.items(), grouped_tif_name.items())

In [ ]:
# initialise results list
satellite_results = []

# initialise dictionary to store dataframes of total counts of points per cluster for example, total count of clusters with 3 points
total_unique_clusters_dfs = {}

# loop through the list of concatenated dataframes and corresponding satellite image .tif files
for (group_1, df), (group_2, tif) in zip(concatenated_csv_dfs.items(), grouped_tif_name.items()):
    # if the keys match
    if group_1 == group_2:
        # open the raster .tif file with rasterio
        tif_file = os.path.join(input_satellite_images, tif)
        with rasterio.open(tif_file) as dataset:
            output_name = tif[:-4]

            # extract the raster CRS information and set the destination crs as 4326
            dst_crs = get_crs(tif_file)
            src_crs = CRS.from_epsg(4326)

            # convert coordinates from the current projection to EPSG:4326
            dataframe = convert_coordinates(df, src_crs, dst_crs)

            # extract the raster geolocation information
            full_arr = get_raster_as_array(tif_file)
            sensor_name = get_raster_sensor(tif_file)
            print(f'sensor_name: {sensor_name}')
            sensor_bands = get_raster_band_count(tif_file)
            print(f'sensor_bands: {sensor_bands}')
            transform = dataset.transform
            origin_x = transform.c
            pixel_width = transform.a
            rotation_x = transform.b
            origin_y = transform.f
            rotation_y = transform.d
            pixel_height = transform.e
            print(f'image size is, {origin_x, pixel_width, rotation_x, origin_y, rotation_y, pixel_height}')
            width = dataset.width
            height = dataset.height
            print(f'Image width: {width}, Image height: {height}')

            # convert georeferenced coordinates (latitude, longitude) into pixel-based coordinates within the image (row, col), using an affine transform
            def latlon_to_pixel(lat, lon, transform):
                '''
                Function to convert georeferenced coordinates (latitude, longitude) into pixel-based coordinates within the image (row, col).
                Uses an affine transform.
                '''
                row, col = ~transform * (lon, lat)
                return int(row), int(col)

            dataframe['pixel_x'], dataframe['pixel_y'] = zip(*dataframe.apply(lambda row: latlon_to_pixel(row['converted_lat'], row['converted_lon'], transform), axis=1))

            # print(dataframe.head())

            # extract gsd and image_id from group_1 (since it represents the keys)
            gsd, image_id = group_1  # group_1 key contains (gsd, image_id)

            # extract the distance threshold and meters per pixel
            # select correct distance_threshold_meters from distance_thresholds dictionary and meters_per_pixel per (gsd, image_id)
            if (gsd, image_id) in distance_thresholds:
                distance_threshold_meters = distance_thresholds[(gsd, image_id)]
            else:
                print(f"Warning: No predefined distance threshold for GSD {gsd}, Image ID {image_id}. Using default value.")
                distance_threshold_meters = 1.8  # default value

            # # define distance_threshold_meters as one value if the same value is applicable to all images
            # distance_threshold_meters = 1.8

            meters_per_pixel = pixel_width #/ width  # adjust based on image width if needed
            # use these values in calculations to convert meters to pixels
            print(f"Distance Threshold (meters): {distance_threshold_meters}")
            print(f"Meters per Pixel: {meters_per_pixel}")

            # convert distance threshold from meters to pixels
            distance_threshold_pixels = distance_threshold_meters / meters_per_pixel
            print(f"Distance Threshold (pixels): {distance_threshold_pixels}")
            # perform clustering with the pixel-based distance threshold
            try:
                clustered_data = perform_clustering(df, distance_threshold_pixels)
            except ValueError as e:
                print(f"Skipping image {tif} due to error: {e}")
                continue
            # print(clustered_data)

            # compute median xy coordinate for each cluster and include the list of 'id' values in each cluster
            consensus_clicks = clustered_data.groupby('cluster').agg({
                'pixel_x': 'median',
                'pixel_y': 'median',
                'id': lambda x: list(x),  # keep a list of the 'id' values for each cluster - important for calculating Rand index
                'unique_id': lambda x: list(x)    # keep a list of 'unique_id' values for each cluster - important for calculating Rand index
            }).reset_index()

            # add a column for the number of points (labels) in each cluster
            cluster_counts = clustered_data.groupby('cluster').size().reset_index(name='label_count')
            # check unique clusters
            unique_clusters = cluster_counts['label_count'].unique()
            print(f'unique_clusters: {unique_clusters}')
            # # if unique_clusters is not equal to the number of observers e.g., 3 observers = [1 2 3]
            # # unhash the following to filter clusters with a label_count of 4 or more to manually check these in the image
            # cluster_with_label_count_4 = cluster_counts[cluster_counts['label_count'] >= 4]
            # # print the clusters exceeding 3
            # print(f'cluster_with_label_count_4: {cluster_with_label_count_4}')

            # unhash to check clusters
            for cluster_id in clustered_data['cluster'].unique():
                cluster_data = clustered_data[clustered_data['cluster'] == cluster_id]
                # print(f'cluster_data: {cluster_data}')

            check_certainty = clustered_data['certainty'].unique()
            # should be equal to ['possible_50-69' 'likely_70-89' 'definite_90-100']
            print(f'check_certainty: {check_certainty}')

            # add a column for the number of definite labels in each cluster
            definite_counts = clustered_data[clustered_data['certainty'] == 'definite_90-100'].groupby('cluster').size().reset_index(name='definite_count')

            # add a column for the number of likely labels in each cluster
            likely_counts = clustered_data[clustered_data['certainty'] == 'likely_70-90'].groupby('cluster').size().reset_index(name='likely_count')

            # add a column for the number of possible labels in each cluster
            possible_counts = clustered_data[clustered_data['certainty'] == 'possible_50-70'].groupby('cluster').size().reset_index(name='possible_count')

            # determine the majority label in each cluster
            majority_label = clustered_data.groupby('cluster').apply(lambda df: df['certainty'].mode()[0] if not df['certainty'].mode().empty else 'Unknown').reset_index(name='majority_label')

            # merge the counts and majority label with consensus_clicks
            consensus_clicks = pd.merge(consensus_clicks, cluster_counts, on='cluster')
            consensus_clicks = pd.merge(consensus_clicks, definite_counts, on='cluster', how='left').fillna(0)
            consensus_clicks = pd.merge(consensus_clicks, likely_counts, on='cluster', how='left').fillna(0)
            consensus_clicks = pd.merge(consensus_clicks, possible_counts, on='cluster', how='left').fillna(0)
            consensus_clicks = pd.merge(consensus_clicks, majority_label, on='cluster')

            # add the image filename to consensus_clicks
            consensus_clicks['tif_filename'] = tif
            satellite_results.append(consensus_clicks)
            # print(consensus_clicks)
            print(len(consensus_clicks['cluster']))

            # convert pixel coordinates to latitude and longitude
            def pixel_to_latlon(pixel_x, pixel_y, transform):
                '''
                Function to transform pixel coordinates to latitude and longitude.
                '''
                lon, lat = transform * (pixel_x, pixel_y)
                return lon, lat

            # the above convert pixel coordinates to latitude and longitude, converts to the image CRS
            #  the below code converts to EPSG:4326
            def transform_to_epsg4326(x, y, raster_crs):
                """
                Transforms coordinates from the raster's native CRS to EPSG:4326.

                Parameters:
                    x (float): X coordinate in raster CRS.
                    y (float): Y coordinate in raster CRS.
                    raster_crs (str): CRS of the raster (e.g., "EPSG:3857").

                Returns:
                    tuple: (longitude, latitude) in EPSG:4326.
                """
                transformer = Transformer.from_crs(raster_crs, "EPSG:4326", always_xy=True)
                lon, lat = transformer.transform(x, y)
                return lon, lat

            # apply the function to convert pixel coordinates to latitude and longitude of the image CRS, to your dataframe
            consensus_clicks['projection_x_coordinate'], consensus_clicks['projection_y_coordinate'] = zip(*consensus_clicks.apply(lambda row: pixel_to_latlon(row['pixel_x'], row['pixel_y'], transform), axis=1))
            print(consensus_clicks)

            # the above applies the function convert pixel coordinates to latitude and longitude of the image CRS, to your dataframe
            # the following code applies the conversion to EPSG:4326
            consensus_clicks[['longitude', 'latitude']] = consensus_clicks.apply(
                lambda row: transform_to_epsg4326(
                    *pixel_to_latlon(row['pixel_x'], row['pixel_y'], transform),  # pixel to raster CRS
                    raster_crs=dst_crs  # Step 2: Raster CRS to EPSG:4326
                ),
                axis=1, result_type='expand'
            )

            # save the consensus_clicks dataframe as a CSV file per gsd_m and img_cat_id
            consensus_clicks.to_csv(os.path.join(output_path, 'clusters', f'{output_name}_predicted.csv'), index=False)
            print(f"Consensus clicks have been saved to f'{output_name}_predicted.csv'")

            # get the total count of unique clusters based on 'label_count'
            total_unique_clusters = consensus_clicks['label_count'].value_counts()
            # convert the series to a dataFrame
            total_unique_clusters_df = total_unique_clusters.reset_index()
            # rename the 'label_count' column to 'point_per_cluster'
            total_unique_clusters_df = total_unique_clusters_df.rename(columns={'label_count': 'point_per_cluster', 'count': 'agreement'})
            # sort the dataFrame by 'point_per_cluster' in descending order
            total_unique_clusters_df = total_unique_clusters_df.sort_values(by='point_per_cluster', ascending=False)
            # add the image filename to the dataFrame
            total_unique_clusters_df['jpg_filename'] = tif
            # store the dataFrame in the dictionary with a unique key
            total_unique_clusters_dfs[file_img] = total_unique_clusters_df
            # display the sorted dataFrame
            print(total_unique_clusters_df)

            # shape markers for different point categories
            markers = ['s', 'o', '^']  # square, circle, triangle

            # stretch the intensity level of the image for optical imagery
            plot_rescaled, sar_image = make_plotting_array(full_arr, sensor_bands, sensor_name, percent_clip=2)
            if plot_rescaled is not None:
                plt.imshow(plot_rescaled)
                plt.title(f'{output_name}')

                for idx, row in clustered_data.iterrows():
                    if row['cluster'] != -1:
                        plt.scatter(row['pixel_x'], row['pixel_y'], color=colorblind_palette['blue'], s=10, marker=markers[2])
                        # plt.text(row['pixel_x'], row['pixel_y'], str(row['cluster']), color='white', fontsize=8)

                # draw consensus cluster medians
                for _, row in consensus_clicks.iterrows():
                    plt.scatter(row['pixel_x'], row['pixel_y'], color=colorblind_palette['yellow'], s=10, marker=markers[1])

                # draw points with at least 3 labels in green
                valid_clusters = consensus_clicks[consensus_clicks['label_count'] >= 3]
                for _, row in valid_clusters.iterrows():
                    plt.scatter(row['pixel_x'], row['pixel_y'], color=colorblind_palette['blueish_green'], s=10, marker=markers[0])

                # add scale bar for distance threshold
                scale_bar_length_pixels = distance_threshold_pixels
                plt.plot([10, 10 + scale_bar_length_pixels], [height - 20, height - 20], color='black', linewidth=3)
                # plt.text(10, height - 40, f'Threshold {distance_threshold_meters} meters', color='black', fontsize=8)

                # remove gridlines
                plt.grid(False)

                # save the annotated image
                output_img_path = os.path.join(output_path, f'{output_name}_satellite_clusters.png')
                plt.savefig(output_img_path, bbox_inches='tight')

                plt.show()
                print(f"Annotation complete for {output_name}. Saved in '{output_path}' folder.")

                plt.close()  # close the figure

            # else if a SAR image do:
            elif sar_image is not None:
                plt.figure(figsize=(10, 10))
                plt.imshow(sar_image, cmap='gray', vmin=0, vmax=988)
                plt.title(f'{output_name}')
                plt.colorbar()

                # draw points with at least 1 labels in vermillion
                for idx, row in clustered_data.iterrows():
                    if row['cluster'] != -1:
                        plt.scatter(row['pixel_x'], row['pixel_y'], color=colorblind_palette['blue'], s=10, marker=markers[2])
                        # plt.text(row['pixel_x'], row['pixel_y'], str(row['cluster']), color='white', fontsize=8)

                # draw points with at least 2 labels in yellow
                for _, row in consensus_clicks.iterrows():
                    plt.scatter(row['pixel_x'], row['pixel_y'], color=colorblind_palette['yellow'], s=10, marker=markers[1])

                # draw points with at least 3 labels in blue
                valid_clusters = consensus_clicks[consensus_clicks['label_count'] >= 3]
                for _, row in valid_clusters.iterrows():
                    plt.scatter(row['pixel_x'], row['pixel_y'], color=colorblind_palette['blueish_green'], s=10, marker=markers[1])

                # add scale bar for distance threshold
                scale_bar_length_pixels = distance_threshold_pixels
                plt.plot([10, 10 + scale_bar_length_pixels], [height - 20, height - 20], color='black', linewidth=3)
                # plt.text(10, height - 40, f'Threshold {distance_threshold_meters} meters', color='white', fontsize=8)

                # remove gridlines
                plt.grid(False)

                # save the annotated image
                output_img_path = os.path.join(output_path, f'{output_name}_satellite_clusters.png')
                plt.savefig(output_img_path, bbox_inches='tight')

                plt.show()
                print(f"Annotation complete for {output_name}. Saved in '{output_path}' folder.")

                plt.close()  # close the figure

### Rand and Adjusted Rand Index

To perform a Rand and Adjusted Rand Index, users must manually review the clustering accuracy (this can be performed in QGIS), and correct the clustering outputs in the .csv files. The original automated clustering .csv should be preserved and any manual corrections should be made in a copy stored in a sub-folder 'manual_correction'. 

In [ ]:
# define the folder path containing the CSV files
cluster_manual_corrections = os.path.join(output_path, 'clusters')

# collect all ground truth and predicted file paths
gt_files = []
pred_files = []

for file in glob.glob(os.path.join(cluster_manual_corrections, "*.csv")):
    if "ground_truth" in file:
        gt_files.append(file)
    elif "predicted" in file:
        pred_files.append(file)

# function to extract the base name (excluding "ground_truth" or "predicted")
# and remove the last underscore before the keyword
def extract_base_name(file_path, keyword):
    '''
    Function to extract the base name, exclusing the value after the final underscore, "ground_truth" or "predicted".
    '''
    base_name = os.path.basename(file_path).replace(keyword, "").replace(".csv", "").strip()
    return re.sub(r'_$', '', base_name.rsplit('_', 1)[0])

# create dictionaries for matching files
gt_dict = {extract_base_name(f, "ground_truth"): f for f in gt_files}
pred_dict = {extract_base_name(f, "predicted"): f for f in pred_files}

# find matched file pairs
matched_files = [(gt_dict[base_name], pred_dict[base_name])
                 for base_name in gt_dict.keys() if base_name in pred_dict]

# initialise a results table
results = []

# process each matched pair
for gt_path, pred_path in matched_files:
    # Read the ground truth and predicted datasets
    gt = pd.read_csv(gt_path, sep=',')
    pred = pd.read_csv(pred_path, sep=',')

    # process the ground truth dataset
    ground_truth_df = pd.DataFrame({'cluster': gt['cluster'], 'unique': gt['unique_id']})
    ground_truth_df['unique'] = ground_truth_df['unique'].apply(lambda x: list(map(int, re.findall(r'\d+', x))))
    ground_truth_expanded = pd.DataFrame(
        [(cluster, unique_id) for cluster, unique_ids in zip(ground_truth_df['cluster'], ground_truth_df['unique'])
         for unique_id in unique_ids], columns=['cluster', 'unique_id']
    )
    ground_truth_expanded = ground_truth_expanded.sort_values(by='unique_id')
    ground_truth = ground_truth_expanded['cluster'].astype(int).values

    # process the predicted dataset
    predict_df = pd.DataFrame({'cluster': pred['cluster'], 'unique': pred['unique_id']})
    predict_df['unique'] = predict_df['unique'].apply(lambda x: list(map(int, re.findall(r'\d+', x))))
    predict_expanded = pd.DataFrame(
        [(cluster, unique_id) for cluster, unique_ids in zip(predict_df['cluster'], predict_df['unique'])
         for unique_id in unique_ids], columns=['cluster', 'unique_id']
    )
    predict_expanded = predict_expanded.sort_values(by='unique_id')
    predicted = predict_expanded['cluster'].astype(int).values

    # calculate Rand index and Adjusted Rand index
    adjusted_rand_idx = adjusted_rand_score(ground_truth, predicted)

    def rand_index_score(labels_true, labels_pred):
        n = len(labels_true)
        a = b = 0
        for i in range(n):
            for j in range(i + 1, n):
                same_cluster_true = labels_true[i] == labels_true[j]
                same_cluster_pred = labels_pred[i] == labels_pred[j]
                if same_cluster_true and same_cluster_pred:
                    a += 1
                elif not same_cluster_true and not same_cluster_pred:
                    b += 1
        total_pairs = comb(n, 2)
        rand_index = (a + b) / total_pairs
        return rand_index

    rand_idx = rand_index_score(ground_truth, predicted)

    print(f"Processing pair:\nGround Truth: {gt_path}\nPredicted: {pred_path}")
    print("Rand Index:", rand_idx)
    print("Adjusted Rand Index:", adjusted_rand_idx)

    # extract the base name for the current file pair
    base_name = extract_base_name(gt_path, "ground_truth")

    # append results to the results list
    results.append({
        "File Name": base_name,
        "Rand Index": rand_idx,
        "Adjusted Rand Index": adjusted_rand_idx
    })

# convert results to a ddataframe
results_df = pd.DataFrame(results)

# print or save the results table
print(results_df)
# optionally, save to a CSV file
results_df.to_csv(os.path.join(cluster_manual_corrections, "rand_index_results.csv"), index=False)

### Other code that may be useful for working with satellite data:

Converting a geospatial optical and SAR image to RGB 8 bit and exporting as png, jpeg or tif.

In [ ]:
# # converting a geospatial optical and SAR image to RGB 8 bit png
# # to produce a jpeg image instead of png, find within the code png and replace with jpeg
# # to produce a tif image instead of png, find within the code png and replace with tif

# # open the .tif image and name the image with the file name less of .tif
# for group, tif in grouped_tif_name.items():
#     tif_file = os.path.join(input_satellite_images, tif)
#     with rasterio.open(tif_file) as dataset:
#         output_name = tif[:-4]

#         # extract the geolocation information
#         full_arr = get_raster_as_array(tif_file)
#         sensor_name = get_raster_sensor(tif_file)
#         print(f'sensor_name: {sensor_name}')
#         sensor_bands = get_raster_band_count(tif_file)
#         print(f'sensor_bands: {sensor_bands}')

#         # stretch the intensity level of the image
#         plot_rescaled = make_plotting_array(full_arr, sensor_bands, sensor_name, percent_clip=2)
#         if plot_rescaled is not None:
#             plt.imshow(plot_rescaled)
#             plt.title(f'{tif}')
#             plt.show()

#             # process and save the image if it's not SAR image with tsx-1 and 1 band
#             arr_8bit = (plot_rescaled * 255).astype(np.uint8)
#             plot_rescaled_out = arr_8bit.transpose(2, 0, 1)
#             plot_rescaled_out.shape

#             # output raster
#             out_tif = os.path.join(input_satellite_images, f'{output_name}.png') # replace png with jpeg for a .jpeg image and tif for a .tif image

#             # copy the metadata
#             out_meta = dataset.meta.copy()

#             # update meta
#             out_meta.update({
#                 "driver": "PNG", # replace PNG with "JPEG" for a .jpeg image and "GTiff" for .tif
#                 "dtype": 'uint8',
#                 "height": plot_rescaled_out.shape[1],
#                 "width": plot_rescaled_out.shape[2],
#                 "transform": dataset.transform,
#                 "count": 3,
#                 "crs": dataset.crs
#             })

#             # write image to PNG file
#             with rasterio.open(out_tif, "w", **out_meta) as dest:
#                 dest.write(plot_rescaled_out.astype('uint8'))
#             print(f'optical image saved as {out_tif}')

#         # else if the image is a SAR image process as follows
#         elif sensor_name == 'tsx-1' and sensor_bands == 1:
#             output_file = os.path.join(input_satellite_images, f'{output_name}.png') # replace png with jpeg for a .jpeg image and tif for a .tif image
#             vmin = 0  # amend accordingly
#             vmax = 988 # amend accordingly

#             with rasterio.open(tif_file) as src:
#                 data = src.read(
#                     out_shape=(src.count, int(src.height), int(src.width)),
#                     resampling=Resampling.bilinear
#                 )
#                 transform = src.transform

#                 # normalise the data to the range [0, 255]
#                 data = np.clip(data, vmin, vmax)
#                 data = ((data - vmin) / (vmax - vmin) * 255).astype(np.uint8)

#             with rasterio.open(output_file, 'w', driver='PNG', height=data.shape[1], width=data.shape[2], count=src.count, dtype=data.dtype) as dst:
#                 dst.write(data) # replace driver='PNG' with driver='JPEG' for a .jpeg image and 'GTiff' for a .tif image

#             print(f'SAR image saved as {output_file} with vmin={vmin} and vmax={vmax}')

## Aerial analysis

### Load aerial count .csv files

In [ ]:
# import all .csv files containing the counts for each observer in aerial imagery
input_aerial_ref_files = glob.glob(os.path.join(input_aerial_counts, '*.csv'))
print(input_aerial_ref_files)

### Load aerial image .jpeg files

In [ ]:
# import all .jpg aerial imagery files
input_aerial_img_files = glob.glob(os.path.join(input_aerial_images, '*.jpg')) # amend .jpg to the relevant format for your image
print(input_aerial_img_files)

### Clean and prepare .csv data

In [ ]:
# access each .csv file and store name in a list
list_aerial_file_name = []
for file in input_aerial_ref_files:
    dir_csv, file_csv = os.path.split(file)
    list_aerial_file_name.append(file_csv)
print(list_aerial_file_name)

# define a function to group .csv filenames by a single separator index
def group_by_index(name, separator = '_', index = 0):
    '''
    Function to group .csv filenames by a single seperator index, corresponding to location.
    '''
    groups = defaultdict(list)
    for n in name:
        parts = n.split(separator)
        key = parts[index] # Use only one index for the key
        groups[key].append(n)
    return groups

# group the strings
grouped_aerial_file_name = group_by_index(list_aerial_file_name, separator = '_', index = 0)
print(grouped_aerial_file_name)

# convert to a list of keys
aerial_group_keys = list(grouped_aerial_file_name.keys())
print(aerial_group_keys)

In [ ]:
# convert grouped strings to a dataframe
aerial_to_open = []
for key, files in grouped_aerial_file_name.items():
    for file in files:
        aerial_to_open.append({'group': key, 'file': file})

aerial_df_list = []
for item in aerial_to_open:
    key = item['group']
    file = item['file']
    df = pd.read_csv(os.path.join(input_aerial_counts, file))
    print(f"df '{file}' created")
    aerial_df_list.append(df)

aerial_df_list_copy = []
for df, item in zip(aerial_df_list, aerial_to_open):
    aerial_df_copy = df.copy()
    print(f"df_copy '{item['file']}' created")
    aerial_df_list_copy.append(aerial_df_copy)

# function to get the value after the last underscore
# Function `get_observer` is imported from strandings_from_space.filenames.
# iterate over the dataframes and add a new column observer and name as observer if the condition is met
for df, item in zip(aerial_df_list_copy, aerial_to_open):
    observer_value = get_observer(item['file'])
    if observer_value == '1':
        # modify the desired column, e.g., change all values in 'column_name' to 'new_value'
        df['observer'] = observer_value
        print(f"Modified DataFrame from file '{item['file']}'")
    elif observer_value == '2':
        # modify the desired column, e.g., change all values in 'column_name' to 'new_value'
        df['observer'] = observer_value
        print(f"Modified DataFrame from file '{item['file']}'")
    elif observer_value == '3':
        # modify the desired column, e.g., change all values in 'column_name' to 'new_value'
        df['observer'] = observer_value
        print(f"Modified DataFrame from file '{item['file']}'")

# function to convert JSON string into a Python dictionary and extract and return the value associated with the 'cx' key
def extract_cx(json_str):
    '''
    Function to convert JSON string into a Python dictionary and extract and return the value associated with the 'cx' key.
    '''
    json_obj = json.loads(json_str)
    return json_obj['cx']

# function to convert JSON string into a Python dictionary and extract and return the value associated with the 'cy' key
def extract_cy(json_str):
    '''
    Function to convert JSON string into a Python dictionary and extract and return the value associated with the 'cy' key.
    '''
    json_obj = json.loads(json_str)
    return json_obj['cy']

# function to extract the second value using regular expressions
def extract_certainty(json_str):
    '''
    Function to extract the second value in the filename corresponding to 'certainty' using regular expressions.
    '''
    match = re.search(r'"(.*?)":"(.*?)"', json_str)
    if match:
        return match.group(2)
    return None

# iterate over the dataframes and add a new column observer and name as observer if the condition is met
for df, item in zip(aerial_df_list_copy, aerial_to_open):
    df['x'] = df['region_shape_attributes'].apply(extract_cx)
    df['y'] = df['region_shape_attributes'].apply(extract_cy)
    df['certainty'] = df['region_attributes'].apply(extract_certainty)
    # convert the 'certainty' column to lowercase
    df['certainty'] = df['certainty'].str.lower()

# print the modified dataframes
for df in aerial_df_list_copy:
    print(df)

### All certainty data

In [ ]:
# create a dictionary to hold lists of dataframe for each group
aerial_grouped_dfs = {}

# iterate over zip(aerial_df_list_cop, aerial_to_open) and append each dataframe to the corresponding group
for df, item in zip(aerial_df_list_copy, aerial_to_open):
    aerial_group = item['group']
    if aerial_group not in aerial_grouped_dfs:
        aerial_grouped_dfs[aerial_group] = []
    aerial_grouped_dfs[aerial_group].append(df)

# use pd.concat to concatenate the dataframes for each group
aerial_concatenated_dfs = {aerial_group: pd.concat(dfs, ignore_index=True) for aerial_group, dfs in aerial_grouped_dfs.items()}

# print the concatenated dataframes
for group, df in aerial_concatenated_dfs.items():
    print(f"Concatenated DataFrame for group '{group}':")
    print(df)

# create a dictionary to hold unique values for each observer
aerial_unique_observer_values = {}

# iterate over concatenated_dfs and get unique values in 'observer' column
for observer, df in aerial_concatenated_dfs.items():
    unique_observer = df['observer'].unique()
    aerial_unique_observer_values[observer] = unique_observer

# print the unique values for each img_id
for observer, unique_observer in aerial_unique_observer_values.items():
    print(f"Unique values in 'group' for observer '{observer}': {unique_observer}")

In [ ]:
# https://datagy.io/pandas-count-unique-values-groupby/
# create a dictionary to store the resulting dataframes
aerial_total_counts_dfs = {}

# iterate over aerial_concatenated_dfs and get unique value counts for each observer within each gsd_m group
for group, df in aerial_concatenated_dfs.items():
    print(f"Unique value counts for group '{group}':")
    aerial_total_counts = df.groupby('filename')['observer'].value_counts()
    aerial_total_counts = aerial_total_counts.sort_index()
    print(aerial_total_counts)
    aerial_total_counts_df = aerial_total_counts.reset_index(name='counts')
    for value in aerial_total_counts_df['filename']:
        aerial_total_counts_df['filename'] = value[:-11]
    # store the dataframe in the dictionary
    aerial_total_counts_dfs[group] = aerial_total_counts_df

print(aerial_total_counts_dfs)

## Plotting

In [ ]:
# set grid style
sns.set(style="whitegrid")
# sns.set(style="white") # Uncomment to remove grid lines if preferred

# iterate over aerial_total_counts_dfs and plot the data for each group
for group, aerial_total_counts_df in aerial_total_counts_dfs.items():
    # calculate the axis maximum for the current dataframe
    axis_max = aerial_total_counts_df['counts'].max() + 10

    # create figure
    plt.figure(figsize=(10, 6))

    # define data to plot
    ax = sns.pointplot(
        x='filename',
        y='counts',
        hue='observer',
        markers=['s', 'o', '^'],
        linestyles=['-', '--', '-.'],
        palette='colorblind',
        data=aerial_total_counts_df,
    )

    # convert the tuple to a string and capitalise the first letter
    group_str = group.capitalize()

    # set the axis limit and amend the axis names
    ax.set(ylim=(0, axis_max))
    plt.xlabel('Aerial Image Location')
    plt.ylabel('Total Count (No. of Individuals)')
    plt.title(f'Plot for location: {group_str} Island')

    # show the plot
    plt.show()

In [ ]:
# for each image catalogue id, get a breadown of total count by certainty for each observer, at each spatial resolution
# https://datagy.io/pandas-count-unique-values-groupby/
# create a dictionary to store the resulting dataframes
aerial_certainty_counts_dfs = {}

# loop over aerial_concatenated_dfs and group data by certainty for each observer within each gsd_m group, to get the count per certainty
for group, df in aerial_concatenated_dfs.items():
    print(f"Unique certainty value counts for group '{group}'")
    aerial_certainty_counts = df.groupby(['filename', 'observer'])['certainty'].value_counts() # get the total count per certainty
    aerial_certainty_counts_df = aerial_certainty_counts.reset_index(name='counts') # reset the column name for certainty counts to counts

    # store the dataframe aerial_certainty_counts_df in the dictionary
    aerial_certainty_counts_df_pivot = aerial_certainty_counts_df.pivot_table(index=['filename', 'observer'], columns='certainty', values='counts', fill_value=0).reset_index()
    aerial_certainty_counts_df_pivot.set_index(['filename', 'observer'], inplace=True)
    print(aerial_certainty_counts_df_pivot)
    aerial_certainty_counts_dfs[group] = aerial_certainty_counts_df_pivot

# for img_id, pivot_df in certainty_counts_dfs.items():

The following cell can be unhashed to acquire dataframes filtered to remove 'possible' detections.<br>

aerial_certainty_counts_dfs_filtered can then be subtituted in the code for plotting and clustering to plot or cluster the filtered data.

In [ ]:
# # create a new dictionary to store filtered dataframes less of 'possible' detections
# aerial_certainty_counts_dfs_filtered = {}

# # Loop through original dictionary and remove the 'possible' column
# for group, df in aerial_certainty_counts_dfs.items():
#     if 'possible' in df.columns:  # Check if 'possible' exists in the dataframe
#         df_filtered = df.drop(columns=['possible'])  # Drop the 'possible' column
#         aerial_certainty_counts_dfs_filtered[group] = df_filtered
#     else:
#         aerial_certainty_counts_dfs_filtered[group] = df  # Keep original if 'possible' doesn't exist

# # display filtered results
# for group, df in aerial_certainty_counts_dfs_filtered.items():
#     print(f"Filtered dataframe for group '{group}':")
#     print(df)

In [ ]:
# assign colors to each observer by certainty level
# for lik ('likely_70-89') and pos ('possible_50-69'), colours are defined as a lighter shade of the colourbline friendly colours selected for def ('definite_90-100') values
observer_colors = {
    '1def': colorblind_palette['blue'],
    '2def': colorblind_palette['orange'],
    '3def': colorblind_palette['blueish_green'],
    '1lik': (0.0, 0.65, 0.85),
    '2lik': (0.93, 0.80, 0.20),
    '3lik': (0.2, 0.76, 0.65),
    '1pos': (0.2, 0.75, 0.95),
    '2pos': (0.95, 0.90, 0.40),
    '3pos': (0.4, 0.86, 0.75)
}

for group, aerial_certainty_counts_df_pivot in aerial_certainty_counts_dfs.items():
    # unstack to separate observers as columns
    df_unstacked = aerial_certainty_counts_df_pivot.unstack(level=-1)  # Unstack the 'observer' column

    # define stacking order (reversed to make "definite" at the bottom)
    stacking_order = ['definite', 'likely', 'possible']

    # initialise figure
    fig, ax = plt.subplots(figsize=(10, 6))

    # plot bars for each observer
    bar_width = 0.25  # width of each observer's bars
    x = range(len(df_unstacked))  # positions for bars (index of the dataframe)
    offsets = [-bar_width, 0, bar_width]  # offset positions for observers

    # initialise list for legend handles and labels
    legend_handles = []
    legend_labels = []

    # iterate over each observer (1, 2, 3) and plot
    for i, observer in enumerate(['1', '2', '3']):
        # Extract the data for the current observer
        observer_df = df_unstacked.xs(observer, level='observer', axis=1)  # Filter by observer

        # map the correct color for each category and observer
        observer_colors_list = [
            observer_colors[f'{observer}def'],  # Color for 'definite'
            observer_colors[f'{observer}lik'],  # Color for 'likely'
            observer_colors[f'{observer}pos'],  # Color for 'possible'
        ]

        # stack bars for the current observer
        bottom_stack = None
        for j, category in enumerate(stacking_order):  # iterate through categories (definite, likely, possible)
            if category in observer_df.columns:
                bars = ax.bar(
                    [p + offsets[i] for p in x],  # adjusted x positions for each observer
                    observer_df[category],  # heights of the bars
                    bar_width,  # width of each bar
                    color=observer_colors_list[j],  # colour for this observer/category
                    bottom=bottom_stack,  # stack bars correctly
                )

                # add the bar handle to legend only once for each observer and category
                legend_handles.append(bars[0])  # add the handle for the first bar in each category
                legend_labels.append(f'Observer {observer} {category}')  # label for each observer and category

                # update the bottom stack for the next category
                bottom_stack = observer_df[category] if bottom_stack is None else bottom_stack + observer_df[category]

    # update y-axis limits based on data
    y_min, y_max = ax.get_ylim()  # get current y-axis limits
    y_limit = y_max + 10
    ax.set_ylim(0, y_limit)

    # set the final legend with proper labels
    ax.legend(handles=legend_handles, labels=legend_labels, bbox_to_anchor=(1.05, 1), loc='upper left')

    # set x-ticks to show actual resolutions
    ax.set_xticks(x)  # set x-ticks to correspond to the index of `df_unstacked`
    # Alternatively, to completely hide x-axis ticks and labels
    ax.tick_params(axis='x', which='both', bottom=False, top=False, labelbottom=False)

    # convert the tuple to a string and capitalise the first letter
    group_str = group.capitalize()

    # amend the axis names and titles
    ax.set_xlabel(f'Location: {group_str} Island')
    ax.set_ylabel('Total Count (No. of Individuals)')
    ax.set_title(f'Plot for aerial image: {group_str} Island')

    # remove gridlines
    ax.grid(False)

    # ensure layout fits neatly
    plt.tight_layout()

    # show plot
    plt.show()

In [ ]:
# plot aerial images to visualise
for img in input_aerial_img_files:
    print(img)
    image = Image.open(img)
    fig, ax = plt.subplots()
    ax.grid(False)  # Remove grid lines
    ax.imshow(image)
plt.show()

In [ ]:
# define marker style based on tool_label
marker_map = {'1':'s', '2': 'o', '3': '^'}

In [ ]:
# list unique observer id to verify they are correct
observer_ids = df['observer'].unique().tolist()
observer_ids

Plot all observers data points on the corresponding aerial image.

In [ ]:
for img in input_aerial_img_files:
    dir_img, file_img = os.path.split(img)
    print(file_img)
    image = Image.open(img)
    fig, ax = plt.subplots()
    ax.imshow(image)
    for group, df in aerial_concatenated_dfs.items():
        if df['filename'].eq(file_img).any():
            # get unique user_ids for the current image and map them to colors
            observer_unique = df['observer'].unique()
            # assign colours to observers
            observer_colors = {
                observer: colorblind_palette[color] for observer, color in zip(observer_ids, ['blue', 'orange', 'blueish_green'])
            }
            print(observer_colors)

            # plot data points for each row in the subset dataframe
            for _, row in df.iterrows():
                x = row['x']
                y = row['y']
                label = row['observer']
                user_color = observer_colors[row['observer']]
                marker = marker_map.get(label, 'o')  # default to circle if label is not found

                # plot points
                ax.scatter(x, y, color=user_color, marker=marker, edgecolor='black', s=25)  # adjust s for marker size

            # remove gridlines
            plt.grid(False)

            # save the annotated image
            output_img_path = os.path.join(output_path, f'{file_img[:-11]}_counts')
            plt.savefig(output_img_path, bbox_inches='tight')

            plt.show()
            # plt.close()

            print(f"Annotation complete for {file_img[:-11]}. Saved in '{output_path}' folder.")

Define the clustering distances to execute Ward's clustering for each aerial image. <br>

For images without a reference, these can be determined using guidance in Supplmental Material S5: found in Clarke et al., (2026): 'Odontocete strandings from space: Accurately counting individuals with very high-resolution optical and synthetic aperture radar satellite imagery'

Pilot whale average sizes based on New Zealand data pers comms (Professor Karen Stockin and Dr Emma Betty).<br>
- Mean length of adult males 5.50 m<br>
- Mean length of adult females 4.32 m<br>
Mean diameter (calculated from on aux girth) to estimate height from the ground:<br>
- Mean height off ground for adult males 0.96 m<br>
- Mean height off ground for adult females 0.73 m <br>

As in aerial imagery we are unable to distinguish sex, an average of both male and female length is used: (550 + 432) / 2 = 4.91 m.<br>

Using ImageJ https://imagej.net/ij/ set scale was used to determine per pixel measurements:<br>
Chatham: distance in pixels 208.6265 known distance 4.91 = 42.4901 (rounded down = 42)<br>
Stewart: distance in pixels 86.8332 known distance 4.91 = 17.6580 (rounded up = 18)<br>

In [ ]:
clustering_distances = os.path.join(input_path, 'clustering_distances.csv')
clustering_distances_df = pd.read_csv(clustering_distances)
clustering_distances_df

In [ ]:
# clustering markings: the script groups stranded cetacean counts from images based on their positions, using a distance threshold to determine proximity.
# observer constraints: ensures each cluster has no more than one marking per observer
# median calculation: calculates the median position for each cluster and identifies clusters with at least three markings.
# results: saves results to a .csv file, including counts of 'definite', 'likely' and 'possible' labels and the majority type within each cluster.
# image annotation: annotates images with clusters, highlights certain clusters in green if they have at least three markings, and includes a scale bar for distance threshold.

# define the parameters for clustering:
def split_clusters_by_constraints(data, max_labels_per_cluster=3):
    '''
    Function to split clusters further by constraints. 
    No cluster should exceed three points and does not contain the same observer more than once.
    Else remove the furthest instance of an observer and reassign to a new cluster.
    '''
    new_cluster_id = data['cluster'].max() + 1  # start new cluster IDs after the existing ones
    print(f'new cluster id is {new_cluster_id}')

    for cluster_id in data['cluster'].unique():
        cluster_data = data[data['cluster'] == cluster_id]

        # calculate the centroid of the cluster
        centroid_x = cluster_data['x'].mean()
        centroid_y = cluster_data['y'].mean()

        # ensure each observer is in a separate cluster within the same cluster group
        for observer, user_group in cluster_data.groupby('observer'):
            if len(user_group) > 1:
                # calculate the distance of each instance from the centroid
                user_group['distance'] = np.sqrt((user_group['x'] - centroid_x)**2 + (user_group['y'] - centroid_y)**2)

                # sort by distance and keep the closest instance
                user_group = user_group.sort_values(by='distance')
                closest_instance_idx = user_group.index[0]

                # reassign all other instances to new clusters
                for idx in user_group.index[1:]:
                    data.loc[idx, 'cluster'] = new_cluster_id
                    new_cluster_id += 1

    return data

# Ffunction to perform clustering with distance threshold
def perform_clustering(data, distance_threshold, max_labels_per_cluster=3):
    '''
    Function to perform hierarchical clustering using Wards method, within a pixel based distance threshold to a maximum of three points per cluster.
    '''
    if data.empty:
        raise ValueError("Data is empty")

    # prepare the coordinates for clustering
    coords = data[['x', 'y']].values

    # perform hierarchical clustering (Ward method)
    linkage_matrix = linkage(coords, method='ward')

    # apply clustering with the pixel-based distance threshold
    clusters = fcluster(linkage_matrix, distance_threshold, criterion='distance')

    # add cluster results to the data
    data['cluster'] = clusters

    # split clusters based on constraints
    data = split_clusters_by_constraints(data, max_labels_per_cluster)

    return data

In [ ]:
# initialise results list
aerial_results = []

# initialise dictionary to store dataframes of total counts of points per cluster for example, total count of clusters with 3 points
total_unique_clusters_dfs_aerial = {}

for img in input_aerial_img_files:
    dir_img, file_img = os.path.split(img)
    print(file_img)
    output_name = file_img[:-4]
    image = Image.open(img)
    width, height = image.size
    fig, ax = plt.subplots()
    ax.imshow(image)
    for group, df in aerial_concatenated_dfs.items():

        aerial_concatenated_distance_df = df.merge(clustering_distances_df, left_on='filename', right_on='jpg_filename', how='left')
        if aerial_concatenated_distance_df['filename'].eq(file_img).any():
            # get the row where the filename matches the file_img
            row = aerial_concatenated_distance_df[aerial_concatenated_distance_df['filename'] == file_img].iloc[0]
            # extract the distance threshold and meters per pixel from the merged dataframe
            distance_threshold_meters = row['distance_threshold_meters']
            meters_per_pixel = row['meters_per_pixel'] / width  # adjust based on image width if needed
            # now use these values in distance_threshold_pixels calculations
            print(f"Distance Threshold (meters): {distance_threshold_meters}")
            print(f"Meters per Pixel: {meters_per_pixel}")

            # convert distance threshold from meters to pixels
            distance_threshold_pixels = distance_threshold_meters / meters_per_pixel
            # perform clustering with the pixel-based distance threshold
            try:
                clustered_data = perform_clustering(df, distance_threshold_pixels)
            except ValueError as e:
                print(f"Skipping image {file_img} due to error: {e}")
                continue
            # print(clustered_data)

            # compute median xy coordinate for each cluster
            consensus_clicks = clustered_data.groupby('cluster').agg({'x': 'median', 'y': 'median'}).reset_index()
            
            # add a column for the number of points (labels) in each cluster
            cluster_counts = clustered_data.groupby('cluster').size().reset_index(name='label_count')
            # # unhash to check clusters
            # unique_clusters = cluster_counts['label_count'].unique()
            # print(f'unique_clusters: {unique_clusters}')
            # # filter clusters with a label_count of 4 or more
            # cluster_with_label_count_4 = cluster_counts[cluster_counts['label_count'] >= 4]
            # # print the clusters with more than 3
            # print(f'cluster_with_label_count_4: {cluster_with_label_count_4}')

            # # unhash to check clusters
            for cluster_id in clustered_data['cluster'].unique():
                cluster_data = clustered_data[clustered_data['cluster'] == cluster_id]
                # print(f'cluster_data: {cluster_data}')

            # check_certainty = clustered_data['certainty'].unique()
            # print(check_certainty)

            # add a column for the number of definite labels in each cluster
            definite_counts = clustered_data[clustered_data['certainty'] == 'definite'].groupby('cluster').size().reset_index(name='definite_count')

            # add a column for the number of likely labels in each cluster
            likely_counts = clustered_data[clustered_data['certainty'] == 'likely'].groupby('cluster').size().reset_index(name='likely_count')

            # add a column for the number of possible labels in each cluster
            possible_counts = clustered_data[clustered_data['certainty'] == 'possible'].groupby('cluster').size().reset_index(name='possible_count')

            # determine the majority label in each cluster
            majority_label = clustered_data.groupby('cluster').apply(lambda df: df['certainty'].mode()[0] if not df['certainty'].mode().empty else 'Unknown').reset_index(name='majority_label')

            # merge the counts and majority label with consensus_clicks
            consensus_clicks = pd.merge(consensus_clicks, cluster_counts, on='cluster')
            consensus_clicks = pd.merge(consensus_clicks, definite_counts, on='cluster', how='left').fillna(0)
            consensus_clicks = pd.merge(consensus_clicks, likely_counts, on='cluster', how='left').fillna(0)
            consensus_clicks = pd.merge(consensus_clicks, possible_counts, on='cluster', how='left').fillna(0)
            consensus_clicks = pd.merge(consensus_clicks, majority_label, on='cluster')

            # add the image filename to consensus_clicks
            consensus_clicks['jpg_filename'] = file_img
            aerial_results.append(consensus_clicks)
            print(f'Consensus_clicks: {consensus_clicks}')
            print(len(consensus_clicks['cluster']))
            unique_clusters = consensus_clicks['label_count'].unique()
            print(f'unique_clusters: {unique_clusters}')

            # save the consensus_clicks dataframe as a CSV file per gsd_m and img_cat_id
            consensus_clicks.to_csv(os.path.join(output_path, 'clusters', f'{output_name}_predicted.csv'), index=False)

            print(f"Consensus clicks have been saved to f'{output_name}_predicted.csv'")
            
            # get the total count of unique clusters based on 'label_count'
            total_unique_clusters = consensus_clicks['label_count'].value_counts()
            # convert the Series to a dataframe
            total_unique_clusters_df = total_unique_clusters.reset_index()
            # rename the 'label_count' column to 'point_per_cluster'
            total_unique_clusters_df = total_unique_clusters_df.rename(columns={'label_count': 'point_per_cluster', 'count': 'agreement'})
            # sort the dataframe by 'point_per_cluster' in descending order
            total_unique_clusters_df = total_unique_clusters_df.sort_values(by='point_per_cluster', ascending=False)
            # add the image filename to the dataframe
            total_unique_clusters_df['jpg_filename'] = file_img
            # store the dataframe in the dictionary with a unique key
            total_unique_clusters_dfs_aerial[file_img] = total_unique_clusters_df
            # display the sorted dataframe
            print(f'total_unique_clusters_df: {total_unique_clusters_df}')

            # draw points and clusters on the image
            draw = ImageDraw.Draw(image)

            #  # draw points with at least 1 labels in red
            # for idx, row in clustered_data.iterrows():
            #     if row['cluster'] != -1:
            #         draw.ellipse((row['x'] - 5, row['y'] - 5, row['x'] + 5, row['y'] + 5), outline='red', fill='red')
            #         draw.text((row['x'], row['y']), str(row['cluster']), fill='white')

            #  # draw points with at least 2 labels in yellow
            # for _, row in consensus_clicks.iterrows():
            #     draw.ellipse((row['x'] - 5, row['y'] - 5, row['x'] + 5, row['y'] + 5), outline='blue', fill='blue')

            #  # draw points with at least 3 labels in blue
            # valid_clusters = consensus_clicks[consensus_clicks['label_count'] >= 3]
            # for _, row in valid_clusters.iterrows():
            #     draw.ellipse((row['x'] - 5, row['y'] - 5, row['x'] + 5, row['y'] + 5), outline='green', fill='green')

            # Reverse mapping: label_count → tool_label
            label_to_tool = {3: '1', 2: '2', 1: '3'}

            # plot points based on label_count
            for _, row in consensus_clicks.iterrows():
                label_count = row['label_count']

                # get the corresponding tool label from reversed mapping
                tool_label = label_to_tool.get(label_count)

                # get marker style using original marker_map
                marker_style = marker_map.get(tool_label, 'o')  # Keep using original marker_map
                color_name = "blue" if label_count == 1 else "yellow" if label_count == 2 else "blueish_green"
                color = colorblind_palette[color_name]

                ax.scatter(row['x'], row['y'], marker=marker_style, color=color, s=2)

            # display the image inline in Jupyter Notebook
            plt.imshow(image)
            plt.axis('off')  # hide axes
            plt.show()

            # add scale bar for distance threshold
            scale_bar_length_pixels = distance_threshold_pixels
            draw.line([(10, height - 20), (10 + scale_bar_length_pixels, height - 20)], fill='black', width=3)
            draw.text((10, height - 40), f'Threshold {distance_threshold_meters} meters', fill='black', anchor='lm')

            # # save the image with annotations
            # output_cluster_path = os.path.join(output_path, f"{file_img[:-4]}_clustered.png")
            # image.save(output_cluster_path)
            # print(f"Annotated image saved to {output_cluster_path}")

            output_cluster_path = os.path.join(output_path, f"{file_img[:-4]}_clustered.png")
            fig.savefig(output_cluster_path, bbox_inches='tight', dpi=300)  # Save the Matplotlib figure
            print(f"Annotated image saved to {output_cluster_path}")

# copyright VGG Image Annotator
# @inproceedings{dutta2019vgg,
#   author = {Dutta, Abhishek and Zisserman, Andrew},
#   title = {The {VIA} Annotation Software for Images, Audio and Video},
#   booktitle = {Proceedings of the 27th ACM International Conference on Multimedia},
#   series = {MM '19},
#   year = {2019},
#   isbn = {978-1-4503-6889-6/19/10},
#   location = {Nice, France},
#   numpages = {4},
#   url = {https://doi.org/10.1145/3343031.3350535},
#   doi = {10.1145/3343031.3350535},
#   publisher = {ACM},
#   address = {New York, NY, USA},
# }

# @misc{dutta2016via,
#   author = "Dutta, A. and Gupta, A. and Zissermann, A.",
#   title = "{VGG} Image Annotator ({VIA})",
#   year = "2016",
#   howpublished = "http://www.robots.ox.ac.uk/~vgg/software/via/",
#   note = "Version: 1.0.6 (https://www.robots.ox.ac.uk/~vgg/software/via/via-1.0.6.html), Accessed: 18 September 2024"
# }

In [ ]:
total_unique_clusters_dfs_aerial

In [ ]:
# calculate the percentage agreement
for filename, df in total_unique_clusters_dfs_aerial.items():
    b = df.loc[df['point_per_cluster'] == 3, 'agreement'].iloc[0]
    c = df.loc[df['point_per_cluster'] == 2, 'agreement'].iloc[0]
    a = df.loc[df['point_per_cluster'] == 1, 'agreement'].iloc[0]
    agreement = a/(b+c)
    percentage_agreement = (1 - agreement) * 100
    print(percentage_agreement)

In [ ]:
#assign colors to observers
point_per_cluster_colors = {
    3: colorblind_palette['blueish_green'],
    2: colorblind_palette['yellow'],
    1: colorblind_palette['blue']
}

for filename, df in total_unique_clusters_dfs_aerial.items():
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(x='point_per_cluster', y='agreement', data=df, hue='point_per_cluster', palette=point_per_cluster_colors)

    # Amend the axis names
    plt.xlabel('Points per Cluster')
    plt.ylabel('Observer Agreement')
    part_filename = filename.split('_')
    location = part_filename[0] # Use only one index for the key
    location = location.capitalize()
    plt.title(f'Plot for: {location} Island')
    plt.tight_layout()
    plt.show()